# Deep Hedging avec coûts de transaction : reproduction et mise à l'épreuve sur un ETF CAC 40

**Projet de Deep Learning — M2 ISF**
**Article de référence :** Brugière, P. & Turinici, G. (2025), *Model-Free Deep Hedging with Transaction Costs and Light Data Requirements*.

## Problématique

Un vendeur de call européen se couvre en achetant et vendant le sous-jacent à 30 dates. Avec des frais de transaction, la couverture « parfaite » de Black-Scholes n'existe plus : chaque rééquilibrage coûte. L'article propose d'**apprendre directement la stratégie de couverture** avec un petit réseau dense, en minimisant la **dispersion du coût final de couverture**, à partir de très peu de données (256 trajectoires).

**Notre question :** *un réseau de neurones apprend-il une couverture plus robuste que le delta de Black-Scholes lorsque les transactions sont coûteuses, et cela tient-il sur de vraies données de marché ?*

## Plan et répartition

| Partie | Contenu | Responsable |
|---|---|---|
| **I** | Reproduction du papier : Black-Scholes, Leland, réseau de neurones ; « faut-il vraiment un réseau ? » | Membre 1 |
| **II** | Scénario concret de courtage (Trade Republic : 1 € par ordre) : grille fréquence de couverture × notionnel | Membre 1 |
| **III** | Généralisation aux régimes de marché (2017, 2021, 2022) sur données historiques | Collègue 1 |
| **IV** | Taille des datasets, chevauchement, ajout de la volatilité et **LSTM** (étude d'ablation) | Collègue 2 |

## Données

- **Simulées** : mouvement brownien géométrique (cadre du papier) pour les parties I et IV.1–IV.2.
- **Réelles** : `data/real_prices.csv` (cours quotidiens 2015–2025 de 38 actions du CAC 40). Le CAC 40 n'étant pas négociable, on construit un **ETF répliquant** fictif à partir de ces actions (section 3).

## Conventions communes (identiques dans tout le notebook)

Call européen fictif **ATM**, vendu par le trader ; prix normalisés $S_0 = K = 1$ ; **30 dates de couverture** ; livraison physique ; frais proportionnels $\alpha$ sur les rééquilibrages et le règlement, **sans frais à l'ouverture** (le delta initial est échangé avec l'acheteur) ; $r = 0$. Les coûts sont exprimés en **% du notionnel initial** : ce ne sont ni des rendements, ni le P&L d'une option cotée.

## Notions de deep learning mobilisées (cours et TP)

Réseau dense et rétropropagation (séance 1) ; séparation train / validation / test et sélection de l'itération sur la validation, comme l'`EarlyStopping` du TP 4 ; optimiseur Adam (séance 3) ; plusieurs graines avec bandes d'incertitude (pistes des TP 1 et 2) ; réseaux récurrents et **LSTM** avec *gradient clipping* (séance 4) ; étude d'ablation.

> **Bibliothèque.** Les TP utilisent Keras ; nous utilisons **PyTorch**, comme l'article. La perte (écart-type du coût de toute une trajectoire de décisions enchaînées) se code plus naturellement avec une boucle d'entraînement explicite qu'avec `model.fit`.

## 0 — Configuration

**Temps de calcul.** Le notebook entraîne environ 220 petits réseaux. Sur un CPU de portable, compter **une à deux heures** pour une exécution complète.

- `QUICK = True` : version rapide (2 graines, 300 itérations) pour vérifier que tout s'exécute. **Ces chiffres ne doivent pas être interprétés.**
- `RECOMPUTE = False` : chaque expérience sauvegarde ses résultats en CSV dans `resultats_projet/`. Une ré-exécution relit les CSV au lieu de ré-entraîner, ce qui permet de modifier une figure sans tout relancer.

**Budget d'entraînement.** On fixe un **nombre de mises à jour de gradient** (2 000, soit 500 époques × 4 batchs de 64 comme dans le papier), et non un nombre d'époques, pour comparer équitablement des jeux de données de tailles différentes.

In [ ]:
from pathlib import Path
import copy
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from torch import nn
from scipy.stats import norm
from IPython.display import display

sns.set(style='whitegrid')
torch.set_num_threads(1)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

QUICK = False        # True : vérification rapide, chiffres non interprétables
RECOMPUTE = False    # True : ré-entraîne même si les CSV existent déjà

N_SEEDS = 2 if QUICK else 5
SEEDS = list(range(N_SEEDS))              # études fines (partie IV)
SEEDS_LIGHT = SEEDS[:2] if QUICK else SEEDS[:3]  # parties I à III
N_UPDATES = 300 if QUICK else 2000   # budget fixe de mises à jour de gradient
EVAL_EVERY = 50                      # évaluation train / validation toutes les 50 mises à jour
BATCH_SIZE = 64
LR = 1e-3

# Cadre simulé du papier
S0, K, MU, SIGMA, R = 1.0, 1.0, 0.05, 0.20, 0.0
N_STEPS = 30          # 30 dates de couverture, donc des fenêtres de 31 prix
T_SIM = 0.25          # maturité 3 mois pour les simulations GBM
T_REAL = N_STEPS / 252  # données réelles : 30 séances = 30/252 année
ALPHA = 0.01          # frais proportionnels de 1 % (niveau du tableau du papier)
ALPHAS = [0.0, 0.002, 0.005, 0.01, 0.02]   # grille de frais du papier

# Scénario de courtage concret (partie II)
FIXED_FEE = 1.0       # Trade Republic : 1 € par ordre
SPREAD = 0.001        # hypothèse : demi-écart achat/vente + glissement de 0,1 % sur un ETF CAC 40 liquide
NOTIONALS = [1_000, 10_000, 100_000]

DATA_FILE = next((p for p in [Path('data/real_prices.csv'), Path('real_prices.csv')] if p.exists()),
                 Path('data/real_prices.csv'))
OUTPUT_DIR = Path('resultats_projet')
OUTPUT_DIR.mkdir(exist_ok=True)

print(f'{N_SEEDS} graines, {N_UPDATES} mises à jour par entraînement, frais α = {ALPHA:.1%}')

## 1 — Interface commune

Toutes les parties utilisent **les mêmes fonctions** : `simulate_gbm`, `bs_call_delta`, `bs_hedging_positions`, `hedging_cost`, `DeltaNN`, `neural_hedging_positions`, `train_hedger`. La fonction de coût suit exactement le code de l'article (call vendu, livraison physique, pas de frais à l'ouverture, $S_0 = K = 1$).

On suit toujours deux indicateurs :

- $\operatorname{écart\text{-}type}(Z)$ : dispersion du coût de couverture, c'est la **fonction de perte** du réseau ;
- $\operatorname{moyenne}(Z)$ : coût moyen, qui n'est **pas** optimisé.

In [ ]:
def simulate_gbm(n_paths, n_steps, S0, mu, sigma, T, seed):
    # Trajectoires indépendantes, shape (n_paths, n_steps+1).
    rng = np.random.default_rng(seed)
    dt = T / n_steps
    Z = rng.standard_normal((n_paths, n_steps))
    log_returns = (mu - 0.5 * sigma**2) * dt + sigma * np.sqrt(dt) * Z
    log_prices = np.column_stack([np.zeros(n_paths), np.cumsum(log_returns, axis=1)])
    return S0 * np.exp(log_prices)


def bs_call_delta(S, K, tau, r, sigma):
    S = np.asarray(S, dtype=float)
    tau = np.asarray(tau, dtype=float)
    tau_safe = np.maximum(tau, 1e-12)
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * tau_safe) / (sigma * np.sqrt(tau_safe))
    return np.where(tau <= 0, np.where(S > K, 1., np.where(S < K, 0., 0.5)), norm.cdf(d1))


def bs_hedging_positions(paths, T, sigma, r=0., K=1.):
    # sigma peut être un nombre ou un tableau (n_paths, n_steps) : volatilité connue à chaque date.
    n = paths.shape[1] - 1
    tau = T - np.arange(n) * T / n
    return bs_call_delta(paths[:, :-1], K, tau[None, :], r, sigma)


def hedging_cost(paths, hedges, alpha, T, r=0., K=1.):
    # Coût actualisé par unité de S0, call physiquement livré, sans frais d'entrée.
    # paths : (batch, n+1), hedges : (batch, n). Différentiable en hedges.
    if paths.ndim != 2 or hedges.shape != (len(paths), paths.shape[1] - 1):
        raise ValueError('Dimensions incompatibles entre prix et positions')
    n = hedges.shape[1]
    prices = paths / paths[:, :1]
    dt = T / n
    times = torch.arange(1, n, device=paths.device, dtype=paths.dtype) * dt
    discounts = torch.exp(-r * times)
    changes = hedges[:, 1:] - hedges[:, :-1]
    intermediate = ((changes + alpha * torch.abs(changes)) * prices[:, 1:n] * discounts).sum(dim=1)
    final_price = prices[:, -1]
    target = (final_price >= K).to(paths.dtype)
    adjustment = target - hedges[:, -1]
    terminal = adjustment * final_price + alpha * torch.abs(adjustment) * final_price - target * K
    return hedges[:, 0] + intermediate + torch.exp(paths.new_tensor(-r * T)) * terminal


def as_tensor(a):
    return torch.as_tensor(a, dtype=torch.float32)


def trading_stats(paths, positions, alpha, T, r=0., K=1., threshold=1e-4):
    n = positions.shape[1]
    prices = paths / paths[:, :1]
    intermediate_trades = positions[:, 1:] - positions[:, :-1]
    target = (prices[:, -1] >= K).to(paths.dtype)
    final_trade = target - positions[:, -1]
    discounts = torch.exp(-r * torch.arange(1, n, dtype=paths.dtype) * T / n)
    fee_mid = alpha * (intermediate_trades.abs() * prices[:, 1:n] * discounts).sum(dim=1)
    fee_end = alpha * final_trade.abs() * prices[:, -1] * np.exp(-r * T)
    return {
        'Frais moyens (%)': 100 * (fee_mid + fee_end).mean().item(),
        'Rééquilibrages moyens': (intermediate_trades.abs() > threshold).float().sum(dim=1).mean().item(),
        'Volume intermédiaire moyen': intermediate_trades.abs().sum(dim=1).mean().item(),
    }


def evaluate(paths_numpy, positions_tensor, alpha, T, r=0., K=1.):
    paths_tensor = as_tensor(paths_numpy)
    with torch.no_grad():
        costs = hedging_cost(paths_tensor, positions_tensor, alpha, T, r=r, K=K)
        stats = trading_stats(paths_tensor, positions_tensor, alpha, T, r=r, K=K)
    return {'Coût moyen (%)': 100 * costs.mean().item(),
            'Écart-type (%)': 100 * costs.std(unbiased=False).item(),
            **stats}


def per_window_metrics(paths_numpy, positions_tensor, alpha, T):
    # Coût, frais et volume pour CHAQUE fenêtre (utile pour les analyses par régime).
    P = as_tensor(paths_numpy)
    with torch.no_grad():
        cost = hedging_cost(P, positions_tensor, alpha, T)
        prices = P / P[:, :1]
        trades = positions_tensor[:, 1:] - positions_tensor[:, :-1]
        final_trade = (prices[:, -1] >= K).to(P.dtype) - positions_tensor[:, -1]
        fees = alpha * ((trades.abs() * prices[:, 1:-1]).sum(dim=1) + final_trade.abs() * prices[:, -1])
        volume = trades.abs().sum(dim=1)
    return pd.DataFrame({'Coût (%)': 100 * cost.numpy(), 'Frais (%)': 100 * fees.numpy(),
                         'Volume': volume.numpy()})


def sliding_windows(prices, n_steps=N_STEPS, stride=1):
    # Découpe une série de prix en fenêtres de n_steps+1 points, chacune normalisée à 1.
    prices = np.asarray(prices, dtype=float)
    starts = np.arange(0, len(prices) - n_steps, stride)
    windows = np.stack([prices[s:s + n_steps + 1] for s in starts])
    return windows / windows[:, :1], starts


def realized_vol(paths, T):
    # Volatilité réalisée annualisée de chaque fenêtre (descriptif seulement).
    n = paths.shape[1] - 1
    return np.std(np.diff(np.log(paths), axis=1), axis=1) * np.sqrt(n / T)

In [ ]:
# Vérifications élémentaires, calculables à la main.
p = as_tensor([[1., 1., 1.], [1., 1.10, 1.20], [1., 0.90, 0.80]])
h0 = torch.zeros((3, 2))
assert torch.allclose(hedging_cost(p, h0, 0., T=1., r=0.), as_tensor([0., 0.20, 0.]))
assert np.isclose(hedging_cost(p[:1], h0[:1], 0.02, T=1., r=0.).item(), 0.02)
assert np.isclose(float(bs_call_delta(1., 1., 0.25, 0., 0.2)), 0.5199, atol=1e-4)

# Le coût calculé fenêtre par fenêtre doit redonner la moyenne de evaluate().
h_test = as_tensor([[0.3, 0.6], [0.5, 0.5], [0.2, 0.1]])
assert np.isclose(per_window_metrics(p.numpy(), h_test, 0.01, 1.)['Coût (%)'].mean(),
                  evaluate(p.numpy(), h_test, 0.01, 1.)['Coût moyen (%)'], atol=1e-5)

# Fenêtres glissantes : 35 prix -> 5 fenêtres de 31 points qui commencent toutes à 1.
w, s = sliding_windows(np.linspace(1, 2, 35), n_steps=30, stride=1)
assert w.shape == (5, 31) and np.allclose(w[:, 0], 1) and list(s) == [0, 1, 2, 3, 4]
print('Tests élémentaires : OK')

### Stratégies classiques : Black-Scholes, Leland et « bande de non-transaction »

- **Black-Scholes** : $H_{t_i} = \Phi(d_1)$ calculé avec la volatilité $\sigma$.
- **Leland (1985)** : même formule, mais avec une volatilité gonflée par les frais (équation (17) de l'article)
$$\nu^{*2} = \sigma^2 + \alpha\,\sigma\sqrt{\frac{2n}{\pi T}}.$$
Un $\nu^*$ plus grand aplatit le delta, réduit le gamma, donc les rééquilibrages et les frais.
- **Bande de non-transaction** : on ne rééquilibre que si le delta BS sort d'une bande $[H_{t_{i-1}} - b,\ H_{t_{i-1}} + b]$, et seulement jusqu'au bord de la bande. Un **seul paramètre** $b$, choisi par recherche sur grille sur les trajectoires d'entraînement. C'est notre réponse à la question du professeur, *« faut-il vraiment un réseau de neurones ? »* : si une règle à un paramètre fait aussi bien que le réseau, le deep learning n'apporte rien ici.

**Pourquoi pas XGBoost ?** Un modèle d'arbres apprend une fonction $X \mapsto y$ à partir de **cibles connues**. Ici il n'y a pas de « bonne position » à imiter : la perte n'est connue qu'**après** les 30 décisions enchaînées (chaque position dépend de la précédente), et on la dérive à travers toute la trajectoire. Cela demande un modèle différentiable entraîné par rétropropagation, ce qu'un réseau permet et un arbre non. On pourrait imiter le delta BS avec XGBoost, mais on ne ferait que reproduire BS.

In [ ]:
def leland_sigma(sigma, alpha, T, n_steps=N_STEPS):
    # Volatilité ajustée de Leland (équation (17) de l'article).
    return np.sqrt(sigma**2 + alpha * sigma * np.sqrt(2 * n_steps / (np.pi * T)))


def band_hedging_positions(paths, T, sigma, band):
    # Suit le delta BS seulement quand il sort de la bande [H - b, H + b].
    target = bs_hedging_positions(paths, T, sigma)
    H = target.copy()
    for i in range(1, target.shape[1]):
        H[:, i] = np.clip(H[:, i - 1], target[:, i] - band, target[:, i] + band)
    return H


def calibrate_band(paths, T, sigma, alpha, grid=np.linspace(0, 0.3, 31)):
    # Choisit la largeur de bande qui minimise l'écart-type du coût sur les trajectoires données.
    stds = [evaluate(paths, as_tensor(band_hedging_positions(paths, T, sigma, b)), alpha, T)['Écart-type (%)']
            for b in grid]
    return float(grid[int(np.argmin(stds))])


def count_orders(paths, positions, threshold=1e-4):
    # Nombre d'ordres passés par fenêtre (rééquilibrages + règlement final), sans l'ouverture.
    prices = paths / paths[:, :1]
    H = np.asarray(positions)
    intermediate = (np.abs(np.diff(H, axis=1)) > threshold).sum(axis=1)
    final = np.abs((prices[:, -1] >= K).astype(float) - H[:, -1]) > threshold
    return intermediate + final


def rebalance_every(positions, k):
    # Ne rééquilibre qu'une séance sur k (la position est conservée entre deux dates).
    H = np.asarray(positions)
    return H[:, (np.arange(H.shape[1]) // k) * k]


assert np.isclose(leland_sigma(0.2, 0., 0.25), 0.2)
assert np.allclose(band_hedging_positions(np.ones((2, 31)), 0.25, 0.2, 0.), bs_hedging_positions(np.ones((2, 31)), 0.25, 0.2))
print('Stratégies classiques : OK')

## 2 — Modèles : MLP du papier, MLP enrichi et LSTM

**MLP (papier).** À chaque date $t_i$, le même réseau dense $3 \to 64 \to 32 \to 1$ reçoit
$X_{t_i} = (S_{t_i}/K,\ T - t_i,\ H_{t_{i-1}})$ et renvoie la nouvelle position $H_{t_i}$. Le réseau n'a **pas de mémoire** : la seule information sur le passé est la position précédente.

**MLP enrichi.** Même architecture avec une 4ᵉ entrée $\hat\sigma_{t_i}$, la volatilité réalisée des 20 derniers rendements quotidiens **connus à la date de décision**.

**LSTM.** On remplace le MLP par une cellule LSTM (séance 4) suivie d'une couche linéaire. Elle reçoit les mêmes entrées, mais garde un **état caché** d'une date à l'autre : elle peut donc, en principe, « se souvenir » de l'agitation passée du prix dans la fenêtre, c'est-à-dire estimer elle-même une volatilité. La taille cachée (22) est choisie pour avoir **le même nombre de paramètres** que le MLP (≈ 2 400), afin que la comparaison ne soit pas faussée par la capacité.

Les deux types de modèles exposent la même méthode `step(features, state)`, ce qui permet d'utiliser **une seule** fonction `hedging_positions` et **une seule** boucle d'entraînement.

In [ ]:
class DeltaNN(nn.Module):
    # MLP du papier : n_inputs -> 64 -> 32 -> 1 (n_inputs = 3 par défaut, 4 avec la volatilité).
    def __init__(self, n_inputs=3):
        super().__init__()
        self.n_inputs = n_inputs
        self.network = nn.Sequential(
            nn.Linear(n_inputs, 64), nn.ReLU(),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, features):
        return self.network(features).squeeze(-1)

    def init_state(self, batch_size):
        return None                      # pas de mémoire

    def step(self, features, state):
        return self.forward(features), None


class DeltaLSTM(nn.Module):
    # Cellule LSTM + couche linéaire. hidden=22 donne le même nombre de paramètres que le MLP.
    def __init__(self, n_inputs=3, hidden=22):
        super().__init__()
        self.n_inputs = n_inputs
        self.hidden = hidden
        self.cell = nn.LSTMCell(n_inputs, hidden)
        self.head = nn.Linear(hidden, 1)

    def init_state(self, batch_size):
        h = torch.zeros(batch_size, self.hidden)
        return (h, h.clone())

    def step(self, features, state):
        h, c = self.cell(features, state)
        return self.head(h).squeeze(-1), (h, c)


def hedging_positions(model, paths, T, K=1., extra=None):
    # Positions (batch, n_steps). extra : information supplémentaire (batch, n_steps), ex. volatilité.
    batch_size, n_steps = paths.shape[0], paths.shape[1] - 1
    prices = paths / paths[:, :1]
    previous_hedge = torch.zeros(batch_size, dtype=paths.dtype)
    state = model.init_state(batch_size)
    positions = []
    for i in range(n_steps):
        columns = [prices[:, i] / K,
                   torch.full_like(previous_hedge, T * (1 - i / n_steps)),
                   previous_hedge]
        if extra is not None:
            columns.append(extra[:, i])
        features = torch.stack(columns, dim=1)
        current_hedge, state = model.step(features, state)
        positions.append(current_hedge)
        previous_hedge = current_hedge
    return torch.stack(positions, dim=1)


def neural_hedging_positions(model, paths, T, K=1.):
    # Interface de la version initiale du projet (3 entrées, sans information supplémentaire).
    return hedging_positions(model, paths, T, K)


def model_positions(model, paths_numpy, T, extra_numpy=None):
    extra = None if extra_numpy is None else as_tensor(extra_numpy)
    with torch.no_grad():
        return hedging_positions(model, as_tensor(paths_numpy), T, extra=extra)


def count_params(model):
    return sum(p.numel() for p in model.parameters())


for name, m in [('MLP (3 entrées)', DeltaNN(3)), ('MLP (4 entrées)', DeltaNN(4)),
                ('LSTM (3 entrées)', DeltaLSTM(3)), ('LSTM (4 entrées)', DeltaLSTM(4))]:
    print(f'{name:18s} : {count_params(m):5d} paramètres')

### Boucle d'entraînement commune `train_hedger`

Elle reçoit les trajectoires, les frais, les hyperparamètres et la graine, puis renvoie le modèle entraîné et son historique d'apprentissage :

1. la **graine** fixe l'initialisation des poids et l'ordre des mini-batchs ;
2. à chaque mise à jour : positions → coûts $Z_i$ → perte $= \operatorname{écart\text{-}type}(Z)$ → `backward()` → *clipping* du gradient (norme ≤ 1) → pas d'Adam ;
3. toutes les `EVAL_EVERY` mises à jour, on mesure l'écart-type sur **tout** le train et sur la validation ;
4. on garde les poids de la **meilleure validation** (équivalent de l'`EarlyStopping(restore_best_weights=True)` de Keras).

In [ ]:
def train_hedger(make_model, train_paths, valid_paths, alpha, T, seed,
                 train_extra=None, valid_extra=None,
                 n_updates=N_UPDATES, batch_size=BATCH_SIZE, lr=LR, eval_every=EVAL_EVERY):
    torch.manual_seed(seed)
    model = make_model()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    generator = torch.Generator().manual_seed(seed)

    X_train, X_valid = as_tensor(train_paths), as_tensor(valid_paths)
    E_train = None if train_extra is None else as_tensor(train_extra)
    E_valid = None if valid_extra is None else as_tensor(valid_extra)
    n = len(X_train)
    batch_size = min(batch_size, n)

    history = []
    best_valid, best_update, best_weights = float('inf'), 0, None
    update = 0
    while update < n_updates:
        for ids in torch.randperm(n, generator=generator).split(batch_size):
            if len(ids) < 2:            # un écart-type sur 1 trajectoire n'a pas de sens
                continue
            model.train()
            batch = X_train[ids]
            extra = None if E_train is None else E_train[ids]
            costs = hedging_cost(batch, hedging_positions(model, batch, T, extra=extra), alpha, T)
            loss = costs.std(unbiased=False)
            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            update += 1

            if update % eval_every == 0 or update == n_updates:
                model.eval()
                with torch.no_grad():
                    train_std = hedging_cost(X_train, hedging_positions(model, X_train, T, extra=E_train),
                                             alpha, T).std(unbiased=False).item()
                    valid_std = hedging_cost(X_valid, hedging_positions(model, X_valid, T, extra=E_valid),
                                             alpha, T).std(unbiased=False).item()
                history.append({'Mise à jour': update, 'Train (%)': 100 * train_std,
                                'Validation (%)': 100 * valid_std})
                if valid_std < best_valid:
                    best_valid, best_update = valid_std, update
                    best_weights = copy.deepcopy(model.state_dict())
            if update >= n_updates:
                break

    model.load_state_dict(best_weights)
    model.eval()
    return model, pd.DataFrame(history), best_update


def fit_and_test(make_model, train_paths, valid_paths, test_paths, T, seed,
                 train_extra=None, valid_extra=None, test_extra=None, alpha=ALPHA):
    # Entraîne, puis mesure train et test avec les poids retenus sur la validation.
    model, history, best_update = train_hedger(make_model, train_paths, valid_paths, alpha, T, seed,
                                               train_extra, valid_extra)
    pos_train = model_positions(model, train_paths, T, train_extra)
    pos_test = model_positions(model, test_paths, T, test_extra)
    row = {'Graine': seed,
           'Meilleure mise à jour': best_update,
           'Écart-type train (%)': evaluate(train_paths, pos_train, alpha, T)['Écart-type (%)'],
           'Écart-type validation (%)': history['Validation (%)'].min(),
           **evaluate(test_paths, pos_test, alpha, T)}
    return model, history, pos_test, row

In [ ]:
def cached(names, compute):
    # Relit les CSV s'ils existent (sauf RECOMPUTE), sinon lance compute() et sauvegarde.
    suffix = '_quick' if QUICK else ''
    files = [OUTPUT_DIR / f'{name}{suffix}.csv' for name in names]
    if not RECOMPUTE and all(f.exists() for f in files):
        print('Résultats déjà calculés, relus depuis :', ', '.join(f.name for f in files))
        return [pd.read_csv(f) for f in files]
    start = time.time()
    frames = compute()
    for frame, f in zip(frames, files):
        frame.to_csv(f, index=False)
    print(f'Calcul terminé en {(time.time() - start) / 60:.1f} min, résultats sauvegardés dans {OUTPUT_DIR}/')
    return frames


def savefig(name):
    suffix = '_quick' if QUICK else ''
    plt.savefig(OUTPUT_DIR / f'{name}{suffix}.png', dpi=150, bbox_inches='tight')


def plot_band(ax, x, values, label, color=None, ls='-'):
    # Moyenne sur les graines ± un écart-type (comme show_results dans les TP).
    values = np.asarray(values, dtype=float)
    mean, std = values.mean(axis=0), values.std(axis=0)
    ax.plot(x, mean, ls=ls, lw=2, label=label, color=color)
    ax.fill_between(x, mean - std, mean + std, color=color, alpha=0.15)


def validation_curves(history, conditions, column='Validation (%)'):
    # Matrice (graines, itérations) pour une configuration donnée.
    h = history
    for key, value in conditions.items():
        h = h[h[key] == value]
    matrix = h.pivot(index='Graine', columns='Mise à jour', values=column)
    return matrix.columns.to_numpy(), matrix.to_numpy()


def seed_summary(results, keys, metrics):
    # Moyenne et écart-type sur les graines.
    table = results.groupby(keys)[metrics].agg(['mean', 'std'])
    table.columns = [f'{m} {"moyen" if s == "mean" else "σ graines"}'.replace('moyen moyen', 'moyen')
                     for m, s in table.columns]
    return table.reset_index()

## 3 — Données réelles : un ETF répliquant le CAC 40

**Construction du sous-jacent.** Le fichier `data/real_prices.csv` contient les cours quotidiens (2015–2025) de 38 actions du CAC 40. Le CAC 40 lui-même est un **indice**, il ne se négocie pas ; on couvre donc un **ETF répliquant** fictif :

- rendement quotidien de l'ETF = moyenne **équipondérée** des rendements des 38 actions (rééquilibrage quotidien), moins des frais de gestion de 0,25 % par an, ordre de grandeur des ETF CAC 40 du marché ;
- les cours du fichier sont a priori des cours **ajustés des dividendes** (type `Adj Close` de Yahoo), donc l'ETF est de type *capitalisant*, comme l'indice *Total Return* utilisé par les auteurs.

Ce n'est **pas** la pondération réelle du CAC 40 (pondération par capitalisation flottante, que le fichier ne permet pas de reconstruire). C'est un proxy négociable et cohérent, présenté comme tel.

**Convention de maturité.** On adapte l'horizon : **30 séances de bourse**, donc $T = 30/252$ année, couverture quotidienne. Le delta de Black-Scholes, la volatilité et le temps restant donné au réseau utilisent tous cette même convention ($r = 0$).

**Volatilité connue à la date de décision.** $\hat\sigma_t$ = écart-type des 20 derniers log-rendements quotidiens (jusqu'à la clôture de $t$ incluse) × $\sqrt{252}$. Aucun rendement futur n'est utilisé.

In [ ]:
raw = pd.read_csv(DATA_FILE, parse_dates=['Date']).set_index('Date').sort_index()
assert raw.notna().all().all() and (raw > 0).all().all(), 'Prix manquants ou négatifs'
print(f'{raw.shape[1]} actions, {len(raw)} séances, du {raw.index[0].date()} au {raw.index[-1].date()}')

ETF_TER = 0.0025   # frais de gestion annuels de l'ETF
VOL_WINDOW = 20    # fenêtre de la volatilité réalisée glissante

stock_returns = raw.pct_change().iloc[1:]
etf_returns = stock_returns.mean(axis=1) - ETF_TER / 252
etf = pd.concat([pd.Series([100.0], index=raw.index[:1]), 100 * (1 + etf_returns).cumprod()])
etf.name = 'ETF CAC 40 (équipondéré)'
log_returns = np.log(etf).diff()
vol_hat = log_returns.rolling(VOL_WINDOW).std() * np.sqrt(252)

fig, axes = plt.subplots(2, 1, figsize=(13, 6.5), sharex=True)
axes[0].plot(etf.index, etf, lw=1.2)
axes[0].set(ylabel='Valeur (base 100)', title='ETF répliquant le CAC 40 (proxy construit à partir de real_prices.csv)')
axes[1].plot(vol_hat.index, 100 * vol_hat, lw=1, color='C3')
axes[1].set(ylabel='σ̂ 20 jours (%)', title='Volatilité réalisée glissante, connue à chaque date')
for ax in axes:
    for start, end, color, label in [('2015-01-01', '2020-12-31', 'C0', 'Entraînement'),
                                      ('2021-01-01', '2021-12-31', 'C1', 'Validation'),
                                      ('2022-01-01', '2025-12-31', 'C2', 'Test')]:
        ax.axvspan(pd.Timestamp(start), pd.Timestamp(end), color=color, alpha=0.07, label=label)
axes[0].legend(loc='upper left')
plt.tight_layout()
savefig('C_etf_cac40')
plt.show()

yearly = pd.DataFrame({
    'Rendement annuel (%)': 100 * etf.groupby(etf.index.year).apply(lambda s: s.iloc[-1] / s.iloc[0] - 1),
    'Volatilité réalisée (%)': 100 * log_returns.groupby(log_returns.index.year).std() * np.sqrt(252),
})
display(yearly.round(1))

### Construction d'un dataset réutilisable

Découpage **chronologique** (on entraîne toujours sur le passé) :

| Rôle | Période | Fenêtres |
|---|---|---|
| Entraînement | 2015 – 2020 | glissantes, pas de 1 séance |
| Validation (choix de l'itération) | 2021 | glissantes, pas de 1 séance |
| Test | 2022 – 2025 | glissantes, pas de 1 séance |

Chaque fenêtre est **entièrement contenue** dans sa période : aucun rendement n'est commun à deux périodes. Seule $\hat\sigma$ peut utiliser des rendements de la période précédente, ce qui est légitime puisqu'ils sont passés. Les fenêtres de test se chevauchent entre elles : les ≈ 1 000 fenêtres ne sont pas 1 000 observations indépendantes (ce sont environ 33 périodes de 30 jours vues avec des dates de départ différentes).

Ce découpage sert aux parties II et IV ; la partie III construit ses propres découpages par année. Chaque fenêtre est normalisée $\widetilde S_{i,t} = S_{i,t}/S_{i,0}$ avec strike $K = 1$ : un **nouveau call ATM** est vendu au début de chaque fenêtre. Le tout est sauvegardé dans `dataset_etf_cac40.npz`, réutilisable par les autres membres du groupe.

In [ ]:
PERIODS = {'train': ('2015-01-01', '2020-12-31'),
           'valid': ('2021-01-01', '2021-12-31'),
           'test':  ('2022-01-01', '2025-12-31')}


def build_real_dataset(prices, vol, start, end, n_steps=N_STEPS, stride=1):
    mask = (prices.index >= start) & (prices.index <= end) & vol.notna().to_numpy()
    p, v, dates = prices[mask].to_numpy(), vol[mask].to_numpy(), prices.index[mask]
    paths, starts = sliding_windows(p, n_steps, stride)
    vols = np.stack([v[s:s + n_steps] for s in starts])   # σ̂ connue aux dates de décision t_0, ..., t_29
    return {'paths': paths, 'vol': vols,
            'start': dates[starts], 'end': dates[starts + n_steps]}


real = {name: build_real_dataset(etf, vol_hat, a, b) for name, (a, b) in PERIODS.items()}
assert real['train']['end'].max() < real['valid']['start'].min()
assert real['valid']['end'].max() < real['test']['start'].min()

np.savez(OUTPUT_DIR / 'dataset_etf_cac40.npz',
         **{f'{k}_{field}': (real[k][field] if field in ('paths', 'vol') else real[k][field].astype(str).to_numpy())
            for k in real for field in ('paths', 'vol', 'start', 'end')})

sigma_train = float(log_returns[PERIODS['train'][0]:PERIODS['train'][1]].std() * np.sqrt(252))
description = pd.DataFrame([{
    'Jeu': k, 'Fenêtres': len(d['paths']),
    'Première fenêtre': d['start'][0].date(), 'Dernière fenêtre': d['end'][-1].date(),
    'Vol. réalisée moyenne des fenêtres (%)': 100 * realized_vol(d['paths'], T_REAL).mean(),
    'σ̂ moyenne au départ (%)': 100 * d['vol'][:, 0].mean(),
    'Part des calls finissant dans la monnaie (%)': 100 * (d['paths'][:, -1] >= 1).mean(),
} for k, d in real.items()])
display(description.round(2))
print(f'Volatilité historique de la période d’entraînement (utilisée par BS à σ fixe) : {100 * sigma_train:.1f} %')

## Partie I — Reproduction du papier (données simulées)

**Objectif.** Retrouver le tableau 1 de l'article : 30 dates de couverture, $T = 0{,}25$, $\sigma = 20\%$, $\mu = 5\%$, $r = 0$, **256 trajectoires d'entraînement**, pour des frais $\alpha \in \{0, 0{,}2, 0{,}5, 1, 2\}\%$.

**Protocole.**
- Entraînement : 256 GBM ; validation (choix de l'itération) : 256 GBM indépendants ; test : **4 096** GBM indépendants (plus que les 256 du papier, pour des estimations plus précises).
- **Un réseau par niveau de frais**, 3 graines chacun.
- Comparaison avec Black-Scholes, Leland et la bande de non-transaction calibrée sur le train.
- On rapporte le **coût moyen** et l'**écart-type** du coût (la perte), sur le train et sur le test.

In [ ]:
train_gbm = simulate_gbm(256, N_STEPS, S0, MU, SIGMA, T_SIM, seed=42)
valid_gbm = simulate_gbm(256, N_STEPS, S0, MU, SIGMA, T_SIM, seed=2025)
test_gbm = simulate_gbm(4096, N_STEPS, S0, MU, SIGMA, T_SIM, seed=2026)

times = np.linspace(0, T_SIM, N_STEPS + 1)
plt.figure(figsize=(10, 4))
for path in train_gbm[:40]:
    plt.plot(times, path, lw=0.8, alpha=0.6)
plt.axhline(K, color='black', ls='--', label='Strike K = 1')
plt.title('40 des 256 trajectoires GBM d’entraînement')
plt.xlabel('Temps (années)')
plt.ylabel('Prix normalisé')
plt.legend()
plt.tight_layout()
savefig('I_trajectoires_gbm')
plt.show()


def compute_P1():
    rows, histories = [], []
    for alpha in ALPHAS:
        band = calibrate_band(train_gbm, T_SIM, SIGMA, alpha)
        classical = {
            'Black-Scholes': lambda p: bs_hedging_positions(p, T_SIM, SIGMA),
            'Leland': lambda p: bs_hedging_positions(p, T_SIM, leland_sigma(SIGMA, alpha, T_SIM)),
            'Bande autour de BS': lambda p: band_hedging_positions(p, T_SIM, SIGMA, band),
        }
        for name, strategy in classical.items():
            for split, paths in [('Train', train_gbm), ('Test', test_gbm)]:
                rows.append({'α (%)': 100 * alpha, 'Stratégie': name, 'Jeu': split, 'Graine': -1,
                             'Paramètre': band if name == 'Bande autour de BS' else np.nan,
                             **evaluate(paths, as_tensor(strategy(paths)), alpha, T_SIM)})
        for seed in SEEDS_LIGHT:
            model, history, best = train_hedger(lambda: DeltaNN(3), train_gbm, valid_gbm, alpha, T_SIM, seed)
            for split, paths in [('Train', train_gbm), ('Test', test_gbm)]:
                rows.append({'α (%)': 100 * alpha, 'Stratégie': 'Deep Hedging (MLP)', 'Jeu': split, 'Graine': seed,
                             'Paramètre': best, **evaluate(paths, model_positions(model, paths, T_SIM), alpha, T_SIM)})
            histories.append(history.assign(**{'α (%)': 100 * alpha, 'Graine': seed}))
            if seed == 0:
                torch.save(model.state_dict(), OUTPUT_DIR / f'I_mlp_alpha{alpha:.3f}{"_quick" if QUICK else ""}.pt')
            print(f'α = {alpha:.1%} | graine {seed} | écart-type test {rows[-1]["Écart-type (%)"]:.3f} %')
    return pd.DataFrame(rows), pd.concat(histories, ignore_index=True)


P1_results, P1_hist = cached(['I_resultats', 'I_historiques'], compute_P1)

In [ ]:
# Tableau au format de l'article : moyenne et écart-type de Z, train et test (moyenne sur les graines pour le réseau).
P1_mean = (P1_results.groupby(['Stratégie', 'Jeu', 'α (%)'])[['Coût moyen (%)', 'Écart-type (%)']].mean()
           .unstack('α (%)'))
P1_mean = P1_mean.reorder_levels([1, 0], axis=1).sort_index(axis=1, level=0)
display(P1_mean.round(2))
P1_mean.to_csv(OUTPUT_DIR / ('I_tableau_papier' + ('_quick' if QUICK else '') + '.csv'))

# Valeurs du tableau 1 de l'article (jeu de test de 256 trajectoires), pour comparaison.
paper_test_std = pd.DataFrame({
    'Black-Scholes':      [0.67, 0.67, 0.76, 0.98, 1.55],
    'Leland':             [0.67, 0.66, 0.70, 0.79, 0.98],
    'Deep Hedging (MLP)': [0.75, 0.78, 0.83, 0.87, 0.97],
}, index=[100 * a for a in ALPHAS])
ours = (P1_results[P1_results['Jeu'] == 'Test'].groupby(['α (%)', 'Stratégie'])['Écart-type (%)'].mean()
        .unstack()[paper_test_std.columns])
comparison = pd.concat({'Article': paper_test_std, 'Nous': ours}, axis=1)
print('Écart-type du coût sur le test (%), article vs reproduction :')
display(comparison.round(2))

In [ ]:
strategy_colors = {'Black-Scholes': 'black', 'Leland': 'C1', 'Bande autour de BS': 'C2', 'Deep Hedging (MLP)': 'C0'}
test = P1_results[P1_results['Jeu'] == 'Test']
summary = test.groupby(['Stratégie', 'α (%)'])[['Écart-type (%)', 'Coût moyen (%)', 'Frais moyens (%)']].agg(['mean', 'std'])

fig, axes = plt.subplots(1, 3, figsize=(17, 4.8))
for name, color in strategy_colors.items():
    s = summary.loc[name]
    for ax, metric in zip(axes, ['Écart-type (%)', 'Coût moyen (%)', 'Frais moyens (%)']):
        ax.errorbar(s.index, s[(metric, 'mean')], yerr=s[(metric, 'std')].fillna(0), marker='o',
                    capsize=3, color=color, label=name)
axes[0].plot(paper_test_std.index, paper_test_std['Deep Hedging (MLP)'], 'x', color='C0', ms=9,
             label='Réseau — article')
for ax, title in zip(axes, ['Risque : écart-type du coût', 'Coût moyen de couverture', 'Frais payés']):
    ax.set(xlabel='Frais proportionnels α (%)', ylabel='% du notionnel', title=title)
axes[0].legend(fontsize=8)
plt.suptitle('Partie I — Test sur 4 096 trajectoires GBM')
plt.tight_layout()
savefig('I_comparaison_frais')
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(16, 4.2), sharey=False)
for ax, alpha in zip(axes, [0.0, 0.01, 0.02]):
    x, values = validation_curves(P1_hist, {'α (%)': 100 * alpha})
    plot_band(ax, x, values, 'Validation', color='C0')
    x, values = validation_curves(P1_hist, {'α (%)': 100 * alpha}, column='Train (%)')
    plot_band(ax, x, values, 'Train', color='C0', ls='--')
    bs_val = evaluate(valid_gbm, as_tensor(bs_hedging_positions(valid_gbm, T_SIM, SIGMA)), alpha, T_SIM)['Écart-type (%)']
    ax.axhline(bs_val, color='black', ls=':', label='Black-Scholes (validation)')
    ax.set(title=f'α = {alpha:.0%}', xlabel='Mises à jour de gradient', ylabel='Écart-type du coût (%)')
    ax.legend(fontsize=8)
plt.suptitle('Partie I — Courbes d’apprentissage (moyenne ± σ sur les graines)')
plt.tight_layout()
savefig('I_courbes_apprentissage')
plt.show()

In [ ]:
# Distribution des coûts sans frais (figure 3 de l'article) et positions sur une trajectoire avec 2 % de frais.
def load_mlp(alpha):
    m = DeltaNN(3)
    m.load_state_dict(torch.load(OUTPUT_DIR / f'I_mlp_alpha{alpha:.3f}{"_quick" if QUICK else ""}.pt'))
    return m.eval()


fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
for alpha, ax in [(0.0, axes[0])]:
    bs_costs = per_window_metrics(test_gbm, as_tensor(bs_hedging_positions(test_gbm, T_SIM, SIGMA)), alpha, T_SIM)['Coût (%)']
    nn_costs = per_window_metrics(test_gbm, model_positions(load_mlp(alpha), test_gbm, T_SIM), alpha, T_SIM)['Coût (%)']
    bins = np.linspace(min(bs_costs.min(), nn_costs.min()), max(bs_costs.max(), nn_costs.max()), 60)
    ax.hist(bs_costs, bins=bins, alpha=0.5, label='Black-Scholes', color='grey')
    ax.hist(nn_costs, bins=bins, alpha=0.5, label='Deep Hedging', color='C0')
    ax.set(xlabel='Coût de couverture Z (% du notionnel)', ylabel='Trajectoires', title='Distribution des coûts, α = 0 %')
    ax.legend()

alpha, idx = 0.02, 0
path = test_gbm[idx:idx + 1]
band = P1_results[(P1_results['Stratégie'] == 'Bande autour de BS') & (P1_results['α (%)'] == 2.0)]['Paramètre'].iloc[0]
curves = {'Black-Scholes': bs_hedging_positions(path, T_SIM, SIGMA)[0],
          'Leland': bs_hedging_positions(path, T_SIM, leland_sigma(SIGMA, alpha, T_SIM))[0],
          'Bande autour de BS': band_hedging_positions(path, T_SIM, SIGMA, band)[0],
          'Deep Hedging (MLP)': model_positions(load_mlp(alpha), path, T_SIM)[0].numpy()}
ax2 = axes[1].twinx()
ax2.plot(times, path[0], color='lightgrey', lw=2, label='Prix (axe de droite)')
ax2.set_ylabel('Prix normalisé')
for name, h in curves.items():
    axes[1].step(times[:-1], h, where='post', color=strategy_colors[name], label=name)
axes[1].set(xlabel='Temps (années)', ylabel='Actions détenues par call', title='Positions sur une trajectoire test, α = 2 %')
axes[1].legend(fontsize=8, loc='upper left')
plt.tight_layout()
savefig('I_distribution_positions')
plt.show()

In [ ]:
# Lecture automatique : le réseau fait-il mieux que les stratégies classiques, et au-delà du bruit des graines ?
s = test.groupby(['α (%)', 'Stratégie'])['Écart-type (%)'].agg(['mean', 'std']).unstack()
for a in s.index:
    nn_mean, nn_std = s.loc[a, ('mean', 'Deep Hedging (MLP)')], s.loc[a, ('std', 'Deep Hedging (MLP)')]
    best_classic = s.loc[a, 'mean'].drop('Deep Hedging (MLP)').idxmin()
    gap = 100 * (1 - nn_mean / s.loc[a, ('mean', best_classic)])
    print(f"α = {a:.1f} % : réseau {nn_mean:.3f} % ± {nn_std:.3f} | meilleure règle classique : {best_classic} "
          f"{s.loc[a, ('mean', best_classic)]:.3f} % | réduction relative du risque : {gap:+.1f} %")

### Commentaire de la partie I (à finaliser après exécution)

- **Ce qu'on attend d'après l'article** : sans frais, le réseau ne bat pas Black-Scholes (c'est la stratégie optimale du modèle, à l'erreur de discrétisation près). L'avantage du réseau apparaît quand $\alpha$ augmente : il apprend à **moins trader**, comme Leland, mais en tenant compte de la position déjà détenue $H_{t-1}$, ce que Leland ne fait pas (remarque 2 de l'article).
- **À vérifier dans nos résultats** : (1) l'ordre de grandeur des écarts-types par rapport au tableau de l'article ; (2) la réduction de risque du réseau par rapport à la **meilleure** règle classique (souvent la bande ou Leland), pas seulement par rapport à BS ; (3) la stabilité entre graines.
- **Faut-il un réseau ?** Si la bande à un paramètre est proche du réseau, la réponse honnête est : *dans un GBM, une règle simple capture l'essentiel ; le réseau devient intéressant quand on veut une règle sans modèle, apprise directement sur des données*. C'est ce que testent les parties III et IV.

## Partie II — Scénario concret : couvrir avec un courtier à 1 € par ordre

**Question du professeur :** *« Si on l'avait fait pour de vrai, qu'est-ce qu'il se serait passé ? »*

**Cadre.** De janvier 2022 à décembre 2025 (période de test), on vend chaque jour un call ATM à 30 séances sur l'ETF CAC 40 et on le couvre via un courtier de type **Trade Republic**, qui facture **1 € par ordre** quel que soit le montant. S'y ajoute un coût proportionnel d'exécution (demi-écart achat/vente et glissement) supposé égal à **0,1 %**. On suppose l'achat de fractions de parts possible.

**Ce que change un frais fixe.** Avec 1 € par ordre, le coût relatif d'un rééquilibrage dépend du **notionnel** : 1 € pèse 0,1 % sur 1 000 € mais 0,001 % sur 100 000 €. On fait donc varier :
- le **notionnel** : 1 000 €, 10 000 €, 100 000 € ;
- la **fréquence de rééquilibrage** de Black-Scholes : tous les 1, 2, 5 ou 10 jours ;
- les stratégies apprises ou calibrées : bande de non-transaction et réseau de neurones.

**Réseau adapté au notionnel.** Le frais fixe n'est pas dérivable (il compte des ordres). On l'approche par un **frais proportionnel équivalent** : $\alpha_{\text{éq}} = 0{,}1\% + 1\,€ / (\text{taille moyenne d'un ordre BS} \times \text{notionnel})$, la taille moyenne étant mesurée sur la période d'entraînement. On entraîne **un réseau par notionnel** sur 2015–2020, et la bande est calibrée de la même façon. L'évaluation, elle, compte les vrais frais : 1 € par ordre + 0,1 %.

In [ ]:
train_real, valid_real, test_real = real['train'], real['valid'], real['test']
bs_train_pos = bs_hedging_positions(train_real['paths'], T_REAL, sigma_train)
trades = np.abs(np.diff(bs_train_pos, axis=1))
typical_trade = trades[trades > 1e-4].mean()
alpha_eq = {n: float(SPREAD + FIXED_FEE / (typical_trade * n)) for n in NOTIONALS}
print(f'Taille moyenne d’un rééquilibrage BS quotidien : {typical_trade:.4f} action par call')
for n, a in alpha_eq.items():
    print(f'Notionnel {n:>7,} € : frais proportionnel équivalent α_éq = {a:.3%}')


def euro_costs(paths, positions, notional):
    # Coût réel en euros : coût proportionnel (SPREAD) + 1 € par ordre.
    H = positions if torch.is_tensor(positions) else as_tensor(positions)
    w = per_window_metrics(paths, H, SPREAD, T_REAL)
    orders = count_orders(paths, H.numpy())
    total = notional * w['Coût (%)'] / 100 + FIXED_FEE * orders
    return pd.DataFrame({'Coût total (€)': total, 'Ordres': orders,
                         'Frais (€)': notional * w['Frais (%)'] / 100 + FIXED_FEE * orders})


def compute_P2():
    rows = []
    for notional in NOTIONALS:
        a = alpha_eq[notional]
        strategies = {f'BS tous les {k} j': as_tensor(rebalance_every(
                          bs_hedging_positions(test_real['paths'], T_REAL, sigma_train), k)) for k in [1, 2, 5, 10]}
        band = calibrate_band(train_real['paths'], T_REAL, sigma_train, a)
        strategies['Bande autour de BS'] = as_tensor(band_hedging_positions(test_real['paths'], T_REAL, sigma_train, band))
        for name, pos in strategies.items():
            e = euro_costs(test_real['paths'], pos, notional)
            rows.append({'Notionnel (€)': notional, 'Stratégie': name, 'Graine': -1, **summarize_euros(e, notional)})
        for seed in SEEDS_LIGHT:
            model, _, _ = train_hedger(lambda: DeltaNN(3), train_real['paths'], valid_real['paths'], a, T_REAL, seed)
            e = euro_costs(test_real['paths'], model_positions(model, test_real['paths'], T_REAL), notional)
            rows.append({'Notionnel (€)': notional, 'Stratégie': 'Deep Hedging (MLP)', 'Graine': seed,
                         **summarize_euros(e, notional)})
            print(f'Notionnel {notional:,} € | graine {seed} | écart-type {rows[-1]["Écart-type (€)"]:.1f} €')
    return [pd.DataFrame(rows)]


def summarize_euros(e, notional):
    return {'Coût moyen (€)': e['Coût total (€)'].mean(),
            'Écart-type (€)': e['Coût total (€)'].std(ddof=0),
            'Écart-type (% notionnel)': 100 * e['Coût total (€)'].std(ddof=0) / notional,
            'Coût 95e centile (€)': e['Coût total (€)'].quantile(0.95),
            'Frais moyens (€)': e['Frais (€)'].mean(),
            'Ordres moyens': e['Ordres'].mean()}


(P2_results,) = cached(['II_trade_republic'], compute_P2)
P2_table = P2_results.groupby(['Notionnel (€)', 'Stratégie'], sort=False).mean(numeric_only=True).drop(columns='Graine')
display(P2_table.round(2))

In [ ]:
fig, axes = plt.subplots(1, len(NOTIONALS), figsize=(17, 4.8))
for ax, notional in zip(axes, NOTIONALS):
    t = P2_table.loc[notional]
    for name, row in t.iterrows():
        color = 'C0' if name.startswith('Deep') else ('C2' if name.startswith('Bande') else 'black')
        ax.scatter(row['Ordres moyens'], row['Écart-type (% notionnel)'], s=60, color=color)
        ax.annotate(name, (row['Ordres moyens'], row['Écart-type (% notionnel)']), fontsize=8,
                    xytext=(4, 4), textcoords='offset points')
    ax.set(title=f'Notionnel {notional:,} €', xlabel='Ordres passés par call (moyenne)',
           ylabel='Écart-type du coût total (% du notionnel)')
plt.suptitle('Partie II — Arbitrage nombre d’ordres / risque avec 1 € par ordre + 0,1 % (test 2022-2025)')
plt.tight_layout()
savefig('II_grille_trade_republic')
plt.show()

for notional in NOTIONALS:
    t = P2_table.loc[notional]
    safest, cheapest = t['Écart-type (€)'].idxmin(), t['Coût moyen (€)'].idxmin()
    print(f'Notionnel {notional:>7,} € :')
    print(f"  la moins risquée  : {safest} (écart-type {t.loc[safest, 'Écart-type (€)']:.1f} €, "
          f"coût moyen {t.loc[safest, 'Coût moyen (€)']:.1f} €, {t.loc[safest, 'Ordres moyens']:.1f} ordres)")
    print(f"  la moins coûteuse : {cheapest} (écart-type {t.loc[cheapest, 'Écart-type (€)']:.1f} €, "
          f"coût moyen {t.loc[cheapest, 'Coût moyen (€)']:.1f} €, {t.loc[cheapest, 'Ordres moyens']:.1f} ordres)")

### Commentaire de la partie II (à finaliser après exécution)

- **Deux critères, deux réponses.** Rééquilibrer moins souvent **réduit les frais et le coût moyen**, mais **augmente le risque** (écart-type), car la couverture colle moins au delta. Pour un petit notionnel, le forfait domine : environ 29 ordres par call avec BS quotidien, soit à peu près 30 € de forfaits sur 1 000 € (3 %). Pour 100 000 €, le forfait devient négligeable. Le bon choix dépend donc de ce que l'on optimise : un vendeur d'options qui veut **maîtriser son risque** et un particulier qui veut **limiter ses frais** ne choisiront pas la même stratégie. Le graphique « ordres / risque » montre cet arbitrage ; une stratégie intéressante est une stratégie qui fait **moins d'ordres sans perdre beaucoup en écart-type** (bande, réseau).
- **Comment le delta hedging change avec les frais** : le réseau entraîné avec un $\alpha_{\text{éq}}$ élevé apprend à faire **moins d'ordres** ; on le voit sur le nombre d'ordres moyen.
- **Limites** : le frais fixe est approché par un frais proportionnel pendant l'entraînement ; le spread de 0,1 % est une hypothèse ; il n'y a pas d'option cotée réellement vendue (la prime encaissée n'est pas modélisée) ; les fenêtres de test se chevauchent.

## Partie III — Généralisation aux régimes de marché (collègue 1)

**Question.** Un réseau entraîné dans un environnement de marché conserve-t-il de bonnes performances dans un autre régime ? C'est une étude de **changement de distribution** (*distribution shift*) entre entraînement et test.

**Trois années de test** aux profils différents (mesurés sur notre ETF ci-dessous) : **2017** (marché calme, faible volatilité), **2021** (forte hausse), **2022** (baisse et volatilité élevée).

**Deux protocoles bien distincts :**

1. **Backtest chronologique (protocole principal).** Pour tester l'année $Y$ : entraînement sur toutes les fenêtres de 2015 à $Y-2$, validation sur $Y-1$, test sur $Y$. On n'utilise **jamais** le futur.
2. **Matrice de transfert rétrospectif (descriptive).** On entraîne sur une seule année $A$ et on teste sur chaque année $B$, y compris $B$ antérieure à $A$. Ce n'est **pas** un backtest : cela mesure seulement à quel point une stratégie apprise dans un régime se transpose à un autre.

**Stratégies comparées** : MLP entraîné sur les fenêtres réelles, MLP entraîné sur GBM simulés (σ = volatilité historique du train), Black-Scholes et Leland à σ historique du train, Black-Scholes à σ̂ glissante. Frais $\alpha = 1\%$.

In [ ]:
REGIMES = [2017, 2021, 2022]
regime_desc = pd.DataFrame([{
    'Année': y,
    'Rendement de l’ETF (%)': yearly.loc[y, 'Rendement annuel (%)'],
    'Volatilité réalisée (%)': yearly.loc[y, 'Volatilité réalisée (%)'],
    'Calls finissant dans la monnaie (%)': 100 * (build_real_dataset(etf, vol_hat, f'{y}-01-01', f'{y}-12-31')['paths'][:, -1] >= 1).mean(),
} for y in REGIMES])
display(regime_desc.round(1))


def compute_P3():
    rows, windows_pos = [], []
    for year in REGIMES:
        train = build_real_dataset(etf, vol_hat, '2015-01-01', f'{year - 2}-12-31')
        valid = build_real_dataset(etf, vol_hat, f'{year - 1}-01-01', f'{year - 1}-12-31')
        test = build_real_dataset(etf, vol_hat, f'{year}-01-01', f'{year}-12-31')
        sig = float(log_returns[:f'{year - 2}-12-31'].std() * np.sqrt(252))
        example = int(np.argsort(realized_vol(test['paths'], T_REAL))[len(test['paths']) // 2])
        strategies = {
            'BS σ historique': as_tensor(bs_hedging_positions(test['paths'], T_REAL, sig)),
            'Leland σ historique': as_tensor(bs_hedging_positions(test['paths'], T_REAL, leland_sigma(sig, ALPHA, T_REAL))),
            'BS σ̂ glissante': as_tensor(bs_hedging_positions(test['paths'], T_REAL, test['vol'])),
        }
        for seed in SEEDS_LIGHT:
            m_real, _, _ = train_hedger(lambda: DeltaNN(3), train['paths'], valid['paths'], ALPHA, T_REAL, seed)
            gbm_train = simulate_gbm(256, N_STEPS, 1., MU, sig, T_REAL, seed=7000 + seed)
            gbm_valid = simulate_gbm(256, N_STEPS, 1., MU, sig, T_REAL, seed=7999)
            m_gbm, _, _ = train_hedger(lambda: DeltaNN(3), gbm_train, gbm_valid, ALPHA, T_REAL, seed)
            strategies[f'MLP données réelles#{seed}'] = model_positions(m_real, test['paths'], T_REAL)
            strategies[f'MLP entraîné sur GBM#{seed}'] = model_positions(m_gbm, test['paths'], T_REAL)
            print(f'Test {year} | graine {seed} : entraînement sur {len(train["paths"])} fenêtres réelles terminé')
        for key, pos in strategies.items():
            name, _, seed = key.partition('#')
            rows.append({'Année test': year, 'Stratégie': name, 'Graine': int(seed) if seed else -1,
                         'Fenêtres train': len(train['paths']), 'Fenêtres test': len(test['paths']),
                         **evaluate(test['paths'], pos, ALPHA, T_REAL)})
            if seed in ('', '0'):
                windows_pos.append(pd.DataFrame({'Année test': year, 'Stratégie': name, 'Date': np.arange(N_STEPS),
                                                 'Position': pos[example].numpy(),
                                                 'Prix': test['paths'][example, :-1]}))
    return pd.DataFrame(rows), pd.concat(windows_pos, ignore_index=True)


P3_results, P3_positions = cached(['III_regimes', 'III_positions'], compute_P3)
P3_table = seed_summary(P3_results, ['Année test', 'Stratégie'],
                        ['Écart-type (%)', 'Coût moyen (%)', 'Frais moyens (%)', 'Volume intermédiaire moyen'])
display(P3_table.round(3))

In [ ]:
regime_colors = {'BS σ historique': 'black', 'Leland σ historique': 'C1', 'BS σ̂ glissante': 'grey',
                 'MLP données réelles': 'C0', 'MLP entraîné sur GBM': 'C4'}
fig, axes = plt.subplots(1, 2, figsize=(15, 4.8))
width = 0.16
for j, name in enumerate(regime_colors):
    t = P3_table[P3_table['Stratégie'] == name].set_index('Année test').loc[REGIMES]
    x = np.arange(len(REGIMES)) + (j - 2) * width
    for ax, metric in zip(axes, ['Écart-type (%)', 'Frais moyens (%)']):
        ax.bar(x, t[f'{metric} moyen'], width, yerr=t[f'{metric} σ graines'].fillna(0), capsize=2,
               color=regime_colors[name], label=name)
for ax, title in zip(axes, ['Risque : écart-type du coût', 'Frais payés']):
    ax.set_xticks(np.arange(len(REGIMES)))
    ax.set_xticklabels([f'{y}\n(vol. {regime_desc.set_index("Année").loc[y, "Volatilité réalisée (%)"]:.0f} %)' for y in REGIMES])
    ax.set(ylabel='% du notionnel', title=title)
axes[0].legend(fontsize=8)
plt.suptitle('Partie III — Backtest chronologique par régime (entraînement antérieur à l’année testée)')
plt.tight_layout()
savefig('III_performance_regimes')
plt.show()

fig, axes = plt.subplots(2, len(REGIMES), figsize=(16, 6.5), sharex=True)
for col, year in enumerate(REGIMES):
    p = P3_positions[P3_positions['Année test'] == year]
    price = p[p['Stratégie'] == 'BS σ historique']['Prix'].to_numpy()
    axes[0, col].plot(price, color='C0')
    axes[0, col].axhline(1, color='grey', ls='--', lw=1)
    axes[0, col].set_title(f'{year} — fenêtre de volatilité médiane')
    for name, color in regime_colors.items():
        q = p[p['Stratégie'] == name]
        axes[1, col].step(q['Date'], q['Position'], where='post', color=color, label=name)
    axes[1, col].set_xlabel('Séance de la fenêtre')
axes[0, 0].set_ylabel('Prix normalisé')
axes[1, 0].set_ylabel('Actions par call')
axes[1, 0].legend(fontsize=7)
plt.suptitle('Partie III — Deltas comparés sur une trajectoire représentative de chaque régime')
plt.tight_layout()
savefig('III_deltas_regimes')
plt.show()

In [ ]:
# Matrice de transfert rétrospectif : entraîné sur l'année A, testé sur l'année B (descriptif, pas un backtest).
# Sans période de validation séparée, l'itération est choisie sur l'année d'entraînement elle-même.
def compute_P3_transfer():
    rows = []
    data = {y: build_real_dataset(etf, vol_hat, f'{y}-01-01', f'{y}-12-31') for y in REGIMES}
    for a in REGIMES:
        for seed in SEEDS_LIGHT:
            model, _, _ = train_hedger(lambda: DeltaNN(3), data[a]['paths'], data[a]['paths'], ALPHA, T_REAL, seed)
            for b in REGIMES:
                rows.append({'Entraîné sur': a, 'Testé sur': b, 'Graine': seed,
                             **evaluate(data[b]['paths'], model_positions(model, data[b]['paths'], T_REAL), ALPHA, T_REAL)})
            print(f'Transfert : entraîné sur {a}, graine {seed}')
    return [pd.DataFrame(rows)]


(P3_transfer,) = cached(['III_transfert'], compute_P3_transfer)
matrix = P3_transfer.groupby(['Entraîné sur', 'Testé sur'])['Écart-type (%)'].mean().unstack()
display(matrix.round(3))

fig, ax = plt.subplots(figsize=(6, 4.5))
sns.heatmap(matrix, annot=True, fmt='.2f', cmap='Blues', ax=ax, cbar_kws={'label': 'Écart-type du coût (%)'})
ax.set_title('Partie III — Transfert rétrospectif entre régimes (MLP)')
plt.tight_layout()
savefig('III_matrice_transfert')
plt.show()

diag = np.diag(matrix.to_numpy())
for b in REGIMES:
    col = matrix[b]
    print(f'Test {b} : entraîné sur la même année {col.loc[b]:.3f} % ; sur une autre année '
          f'{col.drop(b).mean():.3f} % en moyenne (perte due au changement de régime : {col.drop(b).mean() - col.loc[b]:+.3f} pt)')

### Commentaire de la partie III (à finaliser après exécution)

- **À lire** : (1) dans le backtest chronologique, le réseau entraîné sur le passé tient-il face à BS et Leland en 2022, quand la volatilité est bien plus forte qu'en entraînement ? (2) dans la matrice de transfert, la diagonale (même régime) doit être meilleure que le reste si le changement de régime pénalise le réseau ; (3) sur les trajectoires représentatives, le réseau fait-il des deltas plus « paresseux » (moins de rééquilibrages) que BS ?
- **Limites** : environ 220 fenêtres glissantes par année, qui ne contiennent qu'environ 8 périodes de 30 jours indépendantes ; l'ETF est un proxy équipondéré ; un seul niveau de frais. Les écarts sont donc **descriptifs**, pas des tests statistiques.

## Partie IV — Expériences de deep learning (collègue 2)

Quatre études mettent le papier à l'épreuve du point de vue de l'apprentissage :

| Étude | Question | Lien avec l'article |
|---|---|---|
| **IV.1** | 256 trajectoires suffisent-elles ? Un **LSTM** demande-t-il plus de données qu'un MLP ? | « 256 trajectoires suffisent » |
| **IV.2** | Des fenêtres chevauchantes tirées d'une seule trajectoire valent-elles des trajectoires indépendantes, à budget égal ? | Section 6.1 |
| **IV.3** | Combien d'historique réel faut-il ? | Section 6.2 |
| **IV.4** | Ajouter la volatilité $\hat\sigma$ aide-t-il ? Un **LSTM** remplace-t-il cette information ? (ablation) | Limite identifiée par les auteurs |

### IV.1 — Taille du jeu d'entraînement (données simulées)

**Hypothèse.** Les auteurs affirment que 256 trajectoires suffisent. Si c'est vrai, l'écart-type de test doit se stabiliser vers $N = 256$ et la variabilité entre graines doit être faible à partir de là. Pour des petits $N$, on s'attend à un **écart de généralisation** (train ≪ test, sur-apprentissage) et à une forte **instabilité** d'une graine à l'autre.

**Protocole.**
- Tailles $N \in \{32, 64, 128, 256, 512, 1024\}$, **5 graines** par taille. La graine change à la fois les trajectoires d'entraînement simulées et l'initialisation du réseau : la variabilité mesurée inclut donc l'aléa des données et celui de l'optimisation.
- Validation **fixe** : 256 GBM indépendants ; test **fixe** : 4 096 GBM indépendants, jamais utilisés pendant l'entraînement.
- Même budget de 2 000 mises à jour pour toutes les tailles, frais $\alpha = 1\%$, $T = 0{,}25$, 30 dates.
- Deux modèles : MLP du papier et LSTM de même taille.
- **Stabilité des deltas** : pour chaque $(N, \text{modèle})$, on mesure l'écart-type des positions $H_{t_i}$ **entre les graines**, moyenné sur 512 trajectoires de test et sur les 30 dates. Une valeur élevée signifie que deux entraînements « identiques » donnent des stratégies différentes.

In [ ]:
# Même validation (256 GBM) et même test (4 096 GBM) qu'en partie I.
bs_gbm = evaluate(test_gbm, as_tensor(bs_hedging_positions(test_gbm, T_SIM, SIGMA)), ALPHA, T_SIM)
print(f"Référence Black-Scholes sur le test GBM : écart-type {bs_gbm['Écart-type (%)']:.3f} %, "
      f"coût moyen {bs_gbm['Coût moyen (%)']:.3f} %")

SIZES = [32, 64, 128, 256, 512, 1024]
MODELS_B1 = {'MLP': lambda: DeltaNN(3), 'LSTM': lambda: DeltaLSTM(3)}


def compute_B1():
    rows, histories, stability = [], [], []
    for n in SIZES:
        for name, make_model in MODELS_B1.items():
            test_positions = []
            for seed in SEEDS:
                train = simulate_gbm(n, N_STEPS, S0, MU, SIGMA, T_SIM, seed=1000 + seed)
                _, history, pos_test, row = fit_and_test(make_model, train, valid_gbm, test_gbm, T_SIM, seed)
                rows.append({'Modèle': name, 'N': n, **row})
                histories.append(history.assign(**{'Modèle': name, 'N': n, 'Graine': seed}))
                test_positions.append(pos_test[:512].numpy())
                print(f"{name:4s} | N = {n:4d} | graine {seed} | écart-type test {row['Écart-type (%)']:.3f} %")
            stability.append({'Modèle': name, 'N': n,
                              'Dispersion des positions entre graines':
                                  float(np.std(np.stack(test_positions), axis=0).mean())})
    return pd.DataFrame(rows), pd.concat(histories, ignore_index=True), pd.DataFrame(stability)


B1_results, B1_hist, B1_stab = cached(['B1_resultats', 'B1_historiques', 'B1_stabilite'], compute_B1)

In [ ]:
B1_table = seed_summary(B1_results, ['Modèle', 'N'],
                        ['Écart-type (%)', 'Écart-type train (%)', 'Coût moyen (%)', 'Meilleure mise à jour'])
B1_table['Écart de généralisation (%)'] = (B1_table['Écart-type (%) moyen']
                                           - B1_table['Écart-type train (%) moyen'])
B1_table = B1_table.merge(B1_stab, on=['Modèle', 'N'])
display(B1_table.round(3))
B1_table.to_csv(OUTPUT_DIR / ('B1_tableau_stabilite' + ('_quick' if QUICK else '') + '.csv'), index=False)

In [ ]:
colors = {'MLP': 'C0', 'LSTM': 'C3'}
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for name in MODELS_B1:
    g = B1_table[B1_table['Modèle'] == name]
    axes[0].errorbar(g['N'], g['Écart-type (%) moyen'], yerr=g['Écart-type (%) σ graines'],
                     marker='o', capsize=4, color=colors[name], label=f'{name} (test)')
    axes[1].plot(g['N'], g['Écart-type train (%) moyen'], marker='o', color=colors[name], label=f'{name} — train')
    axes[1].plot(g['N'], g['Écart-type (%) moyen'], marker='o', ls='--', color=colors[name], label=f'{name} — test')
axes[0].axhline(bs_gbm['Écart-type (%)'], color='black', ls=':', label='Black-Scholes')
axes[0].axvline(256, color='grey', lw=1, alpha=0.5)
axes[0].set_title('Écart-type du coût sur le test (moyenne ± σ sur les graines)')
axes[1].set_title('Entraînement (trait plein) vs test (pointillés)')
for ax in axes:
    ax.set_xscale('log', base=2)
    ax.set_xticks(SIZES)
    ax.set_xticklabels(SIZES)
    ax.set_xlabel('Nombre de trajectoires d’entraînement N')
    ax.set_ylabel('Écart-type du coût (%)')
    ax.legend()
plt.suptitle(f'B1 — Effet de la taille du jeu d’entraînement (GBM, α = {ALPHA:.0%})')
plt.tight_layout()
savefig('B1_taille_dataset')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), sharey=True)
for ax, n in zip(axes, [32, 256, 1024]):
    for name in MODELS_B1:
        x, values = validation_curves(B1_hist, {'Modèle': name, 'N': n})
        plot_band(ax, x, values, f'{name} — validation', color=colors[name])
        x, values = validation_curves(B1_hist, {'Modèle': name, 'N': n}, column='Train (%)')
        plot_band(ax, x, values, f'{name} — train', color=colors[name], ls='--')
    ax.axhline(bs_gbm['Écart-type (%)'], color='black', ls=':', label='Black-Scholes (test)')
    ax.set_title(f'N = {n} trajectoires')
    ax.set_xlabel('Mises à jour de gradient')
axes[0].set_ylabel('Écart-type du coût (%)')
axes[0].legend(fontsize=8)
plt.suptitle('B1 — Courbes d’apprentissage (moyenne ± σ sur les graines)')
plt.tight_layout()
savefig('B1_courbes_apprentissage')
plt.show()

fig, ax = plt.subplots(figsize=(7, 4))
for name in MODELS_B1:
    g = B1_table[B1_table['Modèle'] == name]
    ax.plot(g['N'], g['Dispersion des positions entre graines'], marker='o', color=colors[name], label=name)
ax.set_xscale('log', base=2)
ax.set_xticks(SIZES)
ax.set_xticklabels(SIZES)
ax.set(xlabel='N', ylabel='σ des positions entre graines (actions)',
       title='B1 — Stabilité des deltas appris')
ax.legend()
plt.tight_layout()
savefig('B1_stabilite_deltas')
plt.show()

In [ ]:
# Lecture automatique des résultats (les chiffres viennent du tableau, rien n'est écrit à l'avance).
for name in MODELS_B1:
    g = B1_table[B1_table['Modèle'] == name].set_index('N')
    ref = g.loc[1024, 'Écart-type (%) moyen']
    enough = [n for n in SIZES if g.loc[n, 'Écart-type (%) moyen'] <= 1.05 * ref]
    print(f'{name} :')
    print(f"  N = 32   : test {g.loc[32, 'Écart-type (%) moyen']:.3f} % ± {g.loc[32, 'Écart-type (%) σ graines']:.3f}, "
          f"écart train/test {g.loc[32, 'Écart de généralisation (%)']:+.3f} pt")
    print(f"  N = 256  : test {g.loc[256, 'Écart-type (%) moyen']:.3f} % ± {g.loc[256, 'Écart-type (%) σ graines']:.3f}, "
          f"écart train/test {g.loc[256, 'Écart de généralisation (%)']:+.3f} pt")
    print(f"  N = 1024 : test {ref:.3f} % ± {g.loc[1024, 'Écart-type (%) σ graines']:.3f}")
    print(f'  Plus petit N à moins de 5 % du résultat obtenu avec 1 024 trajectoires : N = {min(enough)}')
    print(f"  Black-Scholes : {bs_gbm['Écart-type (%)']:.3f} % → le réseau fait mieux que BS dès N = "
          f"{min([n for n in SIZES if g.loc[n, 'Écart-type (%) moyen'] < bs_gbm['Écart-type (%)']], default='jamais')}")

#### Commentaire IV.1 (à finaliser après exécution)

- **Hypothèse** : 256 trajectoires indépendantes suffisent pour apprendre une couverture proche de l'optimum atteignable avec ce réseau.
- **Protocole** : 6 tailles × 2 modèles × 5 graines, budget fixe de 2 000 mises à jour, sélection sur une validation indépendante, test sur 4 096 trajectoires.
- **À lire dans les résultats** : (1) à partir de quel $N$ la courbe de test devient plate ; (2) l'écart train/test, qui mesure le sur-apprentissage ; (3) la largeur de la bande entre graines ; (4) si le LSTM, à paramètres égaux, demande plus de données que le MLP. Dans un GBM, la mémoire n'apporte en théorie rien : les rendements sont indépendants et le delta optimal ne dépend que de $(S/K, \tau, H)$. Un LSTM moins bon serait donc cohérent avec la théorie (optimisation plus difficile, sans information en plus).
- **Limites** : un seul niveau de frais, une seule architecture, données GBM à volatilité constante (le cas le plus favorable au papier).

### IV.2 — Trajectoires indépendantes ou fenêtres chevauchantes (section 6.1)

**Hypothèse.** Dans la section 6.1, les auteurs simulent **une seule** trajectoire de 285 points et en extraient 256 fenêtres glissantes de 30 pas. Ces 256 fenêtres partagent la plupart de leurs rendements : elles contiennent beaucoup moins d'information que 256 trajectoires indépendantes. La question est de savoir si cela dégrade l'apprentissage.

**Budget d'observations.** On compte le nombre de **rendements distincts** vus pendant l'entraînement :

| Jeu d'entraînement | Fenêtres | Rendements distincts |
|---|---|---|
| Indépendantes | $N$ | $30N$ |
| Chevauchantes (1 trajectoire, pas de 1 jour) | $N$ | $N + 29$ |
| Indépendantes, **même budget** que les chevauchantes | $\approx (N+29)/30$ | $\approx N + 29$ |

La troisième ligne est le vrai test : **à quantité de données égale**, vaut-il mieux découper une longue série en fenêtres glissantes ou garder quelques trajectoires indépendantes ?

**Protocole.** $N \in \{64, 256, 1024\}$, 5 graines, MLP du papier, même validation et même test indépendants qu'en IV.1. On mesure aussi la dispersion des volatilités réalisées des fenêtres d'entraînement : des fenêtres chevauchantes couvrent une gamme de volatilités plus étroite.

In [ ]:
OVERLAP_SIZES = [64, 256, 1024]
KINDS = ['Indépendantes', 'Chevauchantes', 'Indépendantes (même budget)']


def make_training_set(kind, n, seed):
    # Renvoie (trajectoires d'entraînement, nombre de rendements distincts).
    if kind == 'Indépendantes':
        return simulate_gbm(n, N_STEPS, S0, MU, SIGMA, T_SIM, seed=2000 + seed), n * N_STEPS
    if kind == 'Chevauchantes':
        n_returns = n + N_STEPS - 1
        long_path = simulate_gbm(1, n_returns, S0, MU, SIGMA, T_SIM * n_returns / N_STEPS, seed=3000 + seed)[0]
        return sliding_windows(long_path, N_STEPS, stride=1)[0], n_returns
    n_paths = max(2, round((n + N_STEPS - 1) / N_STEPS))
    return simulate_gbm(n_paths, N_STEPS, S0, MU, SIGMA, T_SIM, seed=4000 + seed), n_paths * N_STEPS


def compute_B2():
    rows, histories = [], []
    for n in OVERLAP_SIZES:
        for kind in KINDS:
            for seed in SEEDS:
                train, budget = make_training_set(kind, n, seed)
                vols = realized_vol(train, T_SIM)
                _, history, _, row = fit_and_test(lambda: DeltaNN(3), train, valid_gbm, test_gbm, T_SIM, seed)
                rows.append({'Jeu': kind, 'N': n, 'Fenêtres': len(train), 'Rendements distincts': budget,
                             'Vol. réalisée moyenne (%)': 100 * vols.mean(),
                             'Dispersion des vol. réalisées (%)': 100 * vols.std(), **row})
                histories.append(history.assign(**{'Jeu': kind, 'N': n, 'Graine': seed}))
                print(f"{kind:28s} | N = {n:4d} | graine {seed} | écart-type test {row['Écart-type (%)']:.3f} %")
    return pd.DataFrame(rows), pd.concat(histories, ignore_index=True)


B2_results, B2_hist = cached(['B2_resultats', 'B2_historiques'], compute_B2)

In [ ]:
B2_table = seed_summary(B2_results, ['N', 'Jeu', 'Fenêtres', 'Rendements distincts'],
                        ['Écart-type (%)', 'Écart-type train (%)', 'Coût moyen (%)',
                         'Dispersion des vol. réalisées (%)'])
display(B2_table.round(3))

kind_colors = dict(zip(KINDS, ['C0', 'C1', 'C2']))
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for kind in KINDS:
    g = B2_table[B2_table['Jeu'] == kind]
    axes[0].errorbar(g['N'], g['Écart-type (%) moyen'], yerr=g['Écart-type (%) σ graines'],
                     marker='o', capsize=4, color=kind_colors[kind], label=kind)
    axes[1].errorbar(g['Rendements distincts'], g['Écart-type (%) moyen'], yerr=g['Écart-type (%) σ graines'],
                     marker='o', capsize=4, color=kind_colors[kind], label=kind)
for ax in axes:
    ax.axhline(bs_gbm['Écart-type (%)'], color='black', ls=':', label='Black-Scholes')
    ax.set_xscale('log', base=2)
    ax.set_ylabel('Écart-type du coût sur le test (%)')
    ax.legend()
axes[0].set(xlabel='N (valeur nominale de l’expérience)', title='À nombre de fenêtres égal')
axes[1].set(xlabel='Nombre de rendements distincts vus à l’entraînement', title='À budget d’observations égal')
plt.suptitle('B2 — Fenêtres chevauchantes vs trajectoires indépendantes')
plt.tight_layout()
savefig('B2_chevauchement')
plt.show()

for n in OVERLAP_SIZES:
    g = B2_table[B2_table['N'] == n].set_index('Jeu')
    print(f'N = {n} :')
    for kind in KINDS:
        print(f"  {kind:28s} : {g.loc[kind, 'Écart-type (%) moyen']:.3f} % ± {g.loc[kind, 'Écart-type (%) σ graines']:.3f} "
              f"({int(g.loc[kind, 'Rendements distincts'])} rendements distincts, "
              f"dispersion des vol. réalisées {g.loc[kind, 'Dispersion des vol. réalisées (%) moyen']:.2f} pt)")

#### Commentaire IV.2 (à finaliser après exécution)

- **À lire** : (1) à $N$ égal, la perte due au chevauchement (orange vs bleu) ; (2) à **budget égal**, chevauchantes (orange) vs quelques indépendantes (vert) : c'est cette comparaison qui dit si la technique de la section 6.1 est une bonne façon d'utiliser peu de données ; (3) la dispersion des volatilités réalisées, plus faible pour les fenêtres chevauchantes : le réseau voit moins de « scénarios » différents.
- **Lien avec le deep learning** : les fenêtres glissantes ne sont **pas i.i.d.** Les mini-batchs contiennent des exemples très corrélés, ce qui réduit la diversité effective du jeu d'entraînement, un peu comme une augmentation de données qui ne fait que décaler l'image.
- **Limites** : sous GBM, toutes les fenêtres viennent de la même loi. Sur données réelles, une longue série couvre au contraire plusieurs régimes ; c'est ce que testent IV.3 et IV.4.

### IV.3 — Combien d'historique réel faut-il ?

**Hypothèse.** Sur données réelles, plus d'historique apporte plus de régimes de marché (calme 2017, crise Covid 2020…), mais aussi des données plus anciennes, peut-être moins représentatives. La section 6.2 du papier utilise un peu plus d'un an de données (256 fenêtres).

**Protocole.** On prend les $N$ fenêtres **les plus récentes** de la période d'entraînement ($N \in \{32, …, 1024\}$ et la totalité, ≈ 1 450). $N = 256$ correspond à environ un an d'historique, comme dans le papier. MLP du papier, 5 graines (la graine ne change ici que l'initialisation et l'ordre des batchs, les données sont fixées). Validation 2021, test 2022–2025.

**Attention à l'interprétation.** Ici, faire varier $N$ change à la fois la **quantité** de données et la **période** couverte. Les fenêtres les plus récentes contiennent toutes le krach de mars 2020. L'effet observé mélange donc taille et régime.

In [ ]:
# Stratégies de référence sur le test réel (aucun apprentissage).
test_paths, test_vol = real['test']['paths'], real['test']['vol']
baselines = {
    'BS σ fixe (historique train)': as_tensor(bs_hedging_positions(test_paths, T_REAL, sigma_train)),
    'BS σ̂ glissante 20 j': as_tensor(bs_hedging_positions(test_paths, T_REAL, test_vol)),
}
baseline_table = pd.DataFrame([{'Stratégie': name, **evaluate(test_paths, pos, ALPHA, T_REAL)}
                               for name, pos in baselines.items()])
display(baseline_table.round(3))

In [ ]:
REAL_SIZES = [32, 64, 128, 256, 512, 1024, len(real['train']['paths'])]


def compute_C1():
    rows, histories = [], []
    for n in REAL_SIZES:
        train = real['train']['paths'][-n:]
        first_date = real['train']['start'][-n:][0].date()
        for seed in SEEDS:
            _, history, _, row = fit_and_test(lambda: DeltaNN(3), train, real['valid']['paths'],
                                              test_paths, T_REAL, seed)
            rows.append({'N': n, 'Historique depuis': str(first_date), **row})
            histories.append(history.assign(**{'N': n, 'Graine': seed}))
            print(f"N = {n:4d} (depuis {first_date}) | graine {seed} | écart-type test {row['Écart-type (%)']:.3f} %")
    return pd.DataFrame(rows), pd.concat(histories, ignore_index=True)


C1_results, C1_hist = cached(['C1_resultats', 'C1_historiques'], compute_C1)

In [ ]:
C1_table = seed_summary(C1_results, ['N', 'Historique depuis'],
                        ['Écart-type (%)', 'Écart-type train (%)', 'Écart-type validation (%)', 'Coût moyen (%)'])
display(C1_table.round(3))

fig, ax = plt.subplots(figsize=(10, 5))
ax.errorbar(C1_table['N'], C1_table['Écart-type (%) moyen'], yerr=C1_table['Écart-type (%) σ graines'],
            marker='o', capsize=4, label='MLP — test 2022-2025')
ax.plot(C1_table['N'], C1_table['Écart-type train (%) moyen'], marker='o', ls='--', color='C0', alpha=0.6,
        label='MLP — train')
for (name, row), ls in zip(baseline_table.set_index('Stratégie').iterrows(), [':', '-.']):
    ax.axhline(row['Écart-type (%)'], color='black', ls=ls, label=name)
ax.set_xscale('log', base=2)
ax.set_xticks(REAL_SIZES)
ax.set_xticklabels([f"{n}\n{d[:7]}" for n, d in zip(C1_table['N'], C1_table['Historique depuis'])], fontsize=8)
ax.set(xlabel='Nombre de fenêtres d’entraînement (et début de l’historique utilisé)',
       ylabel='Écart-type du coût (%)', title='C1 — Taille de l’historique réel (ETF CAC 40)')
ax.legend(fontsize=8)
plt.tight_layout()
savefig('C1_taille_historique_reel')
plt.show()

### IV.4 — Ajouter la volatilité, et LSTM (étude d'ablation)

**Hypothèse (proposée par les auteurs).** Le réseau original ne connaît pas la volatilité du marché. Sur le S&P 500, ses deltas sont irréalistes « en raison des changements de régime de volatilité ». Lui donner $\hat\sigma$ devrait l'aider à adapter la couverture.

**Configurations comparées** (toutes entraînées sur 2015–2020, validées sur 2021, testées sur 2022–2025, 5 graines, même budget) :

| Configuration | Entrées | Rôle |
|---|---|---|
| MLP (S/K, τ, H) | 3 | modèle original du papier |
| MLP + σ̂ | 4 | modèle enrichi |
| MLP + σ̂ permutée | 4 | **témoin d'ablation** : $\hat\sigma$ est mélangée entre fenêtres. Même nombre de paramètres et même distribution d'entrée, mais aucune information. Si ce modèle gagne autant que « MLP + σ̂ », le gain vient des paramètres, pas de la volatilité |
| LSTM (S/K, τ, H) | 3 | mémoire du passé de la fenêtre : peut-elle remplacer σ̂ ? |
| LSTM + σ̂ | 4 | les deux informations |
| MLP entraîné sur GBM | 3 | approche *model-based* : entraîné sur 256 GBM simulés avec σ = volatilité historique du train |

Références sans apprentissage : **BS à σ fixe** (volatilité historique du train) et **BS à σ̂ glissante**. La seconde est l'équivalent « classique » du MLP + σ̂ : c'est la vraie barre à dépasser.

Les différences du nombre de paramètres sont très faibles (64 de plus pour 4 entrées au lieu de 3) ; le témoin permuté contrôle exactement ce point.

In [ ]:
perm_rng = np.random.default_rng(7)
vol_permuted = {k: real[k]['vol'][perm_rng.permutation(len(real[k]['vol']))] for k in real}


def extra_inputs(kind, split):
    if kind == 'vol':
        return real[split]['vol']
    if kind == 'perm':
        return vol_permuted[split]
    return None


CONFIGS_C2 = {
    'MLP (S/K, τ, H)':             (lambda: DeltaNN(3), None),
    'MLP + σ̂':                    (lambda: DeltaNN(4), 'vol'),
    'MLP + σ̂ permutée (témoin)':  (lambda: DeltaNN(4), 'perm'),
    'LSTM (S/K, τ, H)':            (lambda: DeltaLSTM(3), None),
    'LSTM + σ̂':                   (lambda: DeltaLSTM(4), 'vol'),
    'MLP entraîné sur GBM':        (lambda: DeltaNN(3), 'gbm'),
}
SLUGS = dict(zip(CONFIGS_C2, ['mlp3', 'mlp4_vol', 'mlp4_perm', 'lstm3', 'lstm4_vol', 'mlp3_gbm']))

# Trois fenêtres de test représentatives : σ̂ de départ la plus faible, médiane et la plus forte.
order = np.argsort(test_vol[:, 0])
EXAMPLES = {'Volatilité faible': order[0], 'Volatilité médiane': order[len(order) // 2],
            'Volatilité forte': order[-1]}


def compute_C2():
    rows, histories, windows, positions = [], [], [], []

    def store(name, seed, pos):
        w = per_window_metrics(test_paths, pos, ALPHA, T_REAL)
        windows.append(w.assign(**{'Stratégie': name, 'Graine': seed, 'Fenêtre': np.arange(len(w))}))
        if seed in (-1, 0):
            for label, idx in EXAMPLES.items():
                positions.append(pd.DataFrame({'Stratégie': name, 'Exemple': label, 'Date': np.arange(N_STEPS),
                                               'Position': pos[idx].numpy()}))

    for name, pos in baselines.items():
        store(name, -1, pos)

    for name, (make_model, kind) in CONFIGS_C2.items():
        for seed in SEEDS:
            if kind == 'gbm':
                train = simulate_gbm(256, N_STEPS, 1., MU, sigma_train, T_REAL, seed=5000 + seed)
                valid = simulate_gbm(256, N_STEPS, 1., MU, sigma_train, T_REAL, seed=6000)
                model, history, best_update = train_hedger(make_model, train, valid, ALPHA, T_REAL, seed)
                pos_train = model_positions(model, train, T_REAL)
                pos_test = model_positions(model, test_paths, T_REAL)
                row = {'Graine': seed, 'Meilleure mise à jour': best_update,
                       'Écart-type train (%)': evaluate(train, pos_train, ALPHA, T_REAL)['Écart-type (%)'],
                       'Écart-type validation (%)': history['Validation (%)'].min(),
                       **evaluate(test_paths, pos_test, ALPHA, T_REAL)}
            else:
                model, history, pos_test, row = fit_and_test(
                    make_model, real['train']['paths'], real['valid']['paths'], test_paths, T_REAL, seed,
                    extra_inputs(kind, 'train'), extra_inputs(kind, 'valid'), extra_inputs(kind, 'test'))
            rows.append({'Stratégie': name, 'Paramètres': count_params(model), **row})
            histories.append(history.assign(**{'Stratégie': name, 'Graine': seed}))
            store(name, seed, pos_test)
            if seed == 0:
                torch.save(model.state_dict(), OUTPUT_DIR / f'C2_{SLUGS[name]}{"_quick" if QUICK else ""}.pt')
            print(f"{name:28s} | graine {seed} | écart-type test {row['Écart-type (%)']:.3f} %")

    return (pd.DataFrame(rows), pd.concat(histories, ignore_index=True),
            pd.concat(windows, ignore_index=True), pd.concat(positions, ignore_index=True))


C2_results, C2_hist, C2_windows, C2_positions = cached(
    ['C2_resultats', 'C2_historiques', 'C2_couts_par_fenetre', 'C2_positions_exemples'], compute_C2)

In [ ]:
C2_table = seed_summary(C2_results, ['Stratégie', 'Paramètres'],
                        ['Écart-type (%)', 'Écart-type train (%)', 'Coût moyen (%)',
                         'Frais moyens (%)', 'Volume intermédiaire moyen'])
baseline_rows = baseline_table.rename(columns={'Écart-type (%)': 'Écart-type (%) moyen',
                                               'Coût moyen (%)': 'Coût moyen (%) moyen',
                                               'Frais moyens (%)': 'Frais moyens (%) moyen',
                                               'Volume intermédiaire moyen': 'Volume intermédiaire moyen'})
C2_table = pd.concat([baseline_rows[['Stratégie', 'Écart-type (%) moyen', 'Coût moyen (%) moyen',
                                     'Frais moyens (%) moyen', 'Volume intermédiaire moyen']],
                      C2_table], ignore_index=True)
C2_table = C2_table[['Stratégie', 'Paramètres'] + [c for c in C2_table.columns if c not in ('Stratégie', 'Paramètres')]]
display(C2_table.round(3))
C2_table.to_csv(OUTPUT_DIR / ('C2_tableau_ablation' + ('_quick' if QUICK else '') + '.csv'), index=False)

# Lecture de l'ablation : le gain dépasse-t-il la variabilité entre graines ?
g = C2_results.groupby('Stratégie')['Écart-type (%)'].agg(['mean', 'std'])
def compare(a, b):
    diff = g.loc[a, 'mean'] - g.loc[b, 'mean']
    noise = np.sqrt(g.loc[a, 'std']**2 + g.loc[b, 'std']**2)
    verdict = 'différence nette' if abs(diff) > 2 * noise else 'différence dans le bruit des graines'
    print(f'{a:28s} vs {b:28s} : {diff:+.3f} pt (bruit ≈ {noise:.3f}) → {verdict}')
compare('MLP + σ̂', 'MLP (S/K, τ, H)')
compare('MLP + σ̂', 'MLP + σ̂ permutée (témoin)')
compare('MLP + σ̂ permutée (témoin)', 'MLP (S/K, τ, H)')
compare('LSTM (S/K, τ, H)', 'MLP (S/K, τ, H)')
compare('LSTM + σ̂', 'LSTM (S/K, τ, H)')
compare('MLP (S/K, τ, H)', 'MLP entraîné sur GBM')

In [ ]:
c2_colors = dict(zip(CONFIGS_C2, ['C0', 'C2', 'C7', 'C3', 'C1', 'C4']))
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for name in CONFIGS_C2:
    if name == 'MLP entraîné sur GBM':
        continue    # sa validation est simulée, pas comparable
    x, values = validation_curves(C2_hist, {'Stratégie': name})
    plot_band(axes[0], x, values, name, color=c2_colors[name])
axes[0].set(xlabel='Mises à jour de gradient', ylabel='Écart-type validation 2021 (%)',
            title='Courbes de validation (moyenne ± σ sur les graines)')
axes[0].legend(fontsize=8)

order_names = list(baselines) + list(CONFIGS_C2)
t = C2_table.set_index('Stratégie').loc[order_names]
axes[1].barh(range(len(t)), t['Écart-type (%) moyen'], xerr=t['Écart-type (%) σ graines'].fillna(0),
             color=['grey', 'grey'] + [c2_colors[n] for n in CONFIGS_C2], capsize=3)
axes[1].set_yticks(range(len(t)))
axes[1].set_yticklabels(order_names)
axes[1].invert_yaxis()
axes[1].set(xlabel='Écart-type du coût sur le test 2022-2025 (%)', title='Performance hors échantillon')
plt.suptitle(f'C2 — Ablation sur l’ETF CAC 40 (α = {ALPHA:.0%})')
plt.tight_layout()
savefig('C2_ablation')
plt.show()

#### Dans quels cas la volatilité aide-t-elle ?

On découpe les fenêtres de test (1) par **année** et (2) par **tercile de $\hat\sigma$ au moment de la vente du call** (régime connu ex ante). Pour chaque stratégie et chaque graine, on calcule l'écart-type des coûts dans chaque groupe, puis on moyenne sur les graines.

In [ ]:
groups = pd.DataFrame({'Fenêtre': np.arange(len(test_paths)),
                       'Année': real['test']['start'].year,
                       'Régime de vol. au départ': pd.qcut(test_vol[:, 0], 3,
                                                           labels=['σ̂ faible', 'σ̂ moyenne', 'σ̂ forte'])})
detail = C2_windows.merge(groups, on='Fenêtre')
show = list(baselines) + ['MLP (S/K, τ, H)', 'MLP + σ̂', 'MLP + σ̂ permutée (témoin)', 'LSTM (S/K, τ, H)', 'LSTM + σ̂']

for group_col in ['Régime de vol. au départ', 'Année']:
    table = (detail.groupby(['Stratégie', 'Graine', group_col], observed=True)['Coût (%)']
             .std(ddof=0).groupby(['Stratégie', group_col], observed=True).mean()
             .unstack(group_col).loc[show])
    print(f'Écart-type du coût (%) par {group_col.lower()} :')
    display(table.round(3))

regime_table = (detail.groupby(['Stratégie', 'Graine', 'Régime de vol. au départ'], observed=True)['Coût (%)']
                .std(ddof=0).groupby(['Stratégie', 'Régime de vol. au départ'], observed=True).mean()
                .unstack().loc[show])
ax = regime_table.T.plot.bar(figsize=(12, 5), rot=0,
                             color=['grey', 'silver'] + [c2_colors[n] for n in show[2:]])
ax.set(ylabel='Écart-type du coût (%)', xlabel='', title='C2 — Performance selon le régime de volatilité au départ')
ax.legend(fontsize=8)
plt.tight_layout()
savefig('C2_par_regime')
plt.show()

In [ ]:
# Positions sur les trois fenêtres de test représentatives (graine 0).
show_pos = ['BS σ fixe (historique train)', 'BS σ̂ glissante 20 j', 'MLP (S/K, τ, H)', 'MLP + σ̂', 'LSTM (S/K, τ, H)']
pos_colors = dict(zip(show_pos, ['black', 'grey', 'C0', 'C2', 'C3']))
fig, axes = plt.subplots(2, 3, figsize=(16, 7), sharex=True)
for col, (label, idx) in enumerate(EXAMPLES.items()):
    axes[0, col].plot(test_paths[idx], color='C0')
    axes[0, col].axhline(1, color='grey', ls='--', lw=1)
    axes[0, col].set_title(f"{label} — départ {real['test']['start'][idx].date()}, σ̂₀ = {100 * test_vol[idx, 0]:.0f} %")
    for name in show_pos:
        p = C2_positions[(C2_positions['Stratégie'] == name) & (C2_positions['Exemple'] == label)]
        axes[1, col].step(p['Date'], p['Position'], where='post', label=name, color=pos_colors[name],
                          ls='--' if name.startswith('BS') else '-')
    axes[1, col].set_xlabel('Séance de la fenêtre')
axes[0, 0].set_ylabel('Prix normalisé S/S₀')
axes[1, 0].set_ylabel('Actions détenues par call')
axes[1, 0].legend(fontsize=8)
plt.suptitle('C2 — Couvertures sur trois fenêtres réelles de l’ETF CAC 40')
plt.tight_layout()
savefig('C2_positions_exemples')
plt.show()

#### Le réseau a-t-il appris le rôle de la volatilité ?

En Black-Scholes, une volatilité plus forte **aplatit** le delta autour du strike. On interroge directement le « MLP + σ̂ » (graine 0) sur une grille de prix, à mi-maturité, en lui donnant comme position précédente le delta BS correspondant, pour trois valeurs de $\hat\sigma$. Si les courbes du réseau s'écartent les unes des autres comme celles de BS, le réseau utilise réellement l'information de volatilité.

In [ ]:
mlp_vol = DeltaNN(4)
mlp_vol.load_state_dict(torch.load(OUTPUT_DIR / f'C2_mlp4_vol{"_quick" if QUICK else ""}.pt'))
mlp_vol.eval()

grid = np.linspace(0.85, 1.15, 121)
tau = T_REAL / 2
fig, ax = plt.subplots(figsize=(9, 5))
for sig, color in zip([0.10, 0.20, 0.35], ['C0', 'C1', 'C3']):
    bs_delta = bs_call_delta(grid, 1., tau, 0., sig)
    features = as_tensor(np.column_stack([grid, np.full_like(grid, tau), bs_delta, np.full_like(grid, sig)]))
    with torch.no_grad():
        nn_delta = mlp_vol(features).numpy()
    ax.plot(grid, nn_delta, color=color, lw=2, label=f'MLP + σ̂, σ̂ = {sig:.0%}')
    ax.plot(grid, bs_delta, color=color, ls='--', lw=1, label=f'Black-Scholes, σ = {sig:.0%}')
ax.axvline(1, color='grey', lw=1)
ax.set(xlabel='S/K', ylabel='Position (actions par call)',
       title=f'Delta appris en fonction de σ̂ (τ = {tau * 252:.0f} séances restantes)')
ax.legend(fontsize=8)
plt.tight_layout()
savefig('C2_delta_selon_volatilite')
plt.show()

#### Scénario concret : un call de 10 000 € de notionnel

Si l'on avait vendu, chaque jour de 2022 à 2025, un call ATM à 30 séances sur 10 000 € d'ETF CAC 40 et qu'on l'avait couvert avec chaque stratégie, que se serait-il passé ? Les coûts sont exprimés en euros. Le 95ᵉ centile représente un « mauvais cas » (1 fenêtre sur 20 coûte plus cher). Toutes les graines sont mises en commun pour les stratégies apprises.

In [ ]:
NOTIONAL = 10_000
euros = (C2_windows.groupby('Stratégie')
         .agg(**{'Coût moyen (€)': ('Coût (%)', 'mean'),
                 'Écart-type (€)': ('Coût (%)', lambda x: x.std(ddof=0)),
                 'Coût 95e centile (€)': ('Coût (%)', lambda x: x.quantile(0.95)),
                 'Frais moyens (€)': ('Frais (%)', 'mean')})
         .mul(NOTIONAL / 100).loc[list(baselines) + list(CONFIGS_C2)])
display(euros.round(1))
euros.to_csv(OUTPUT_DIR / ('C2_scenario_euros' + ('_quick' if QUICK else '') + '.csv'))
print('Le coût moyen inclut le paiement du call à l’acheteur : il n’est pas un « profit ». '
      'La prime encaissée à la vente n’est pas modélisée ; c’est la dispersion qui mesure la qualité de la couverture.')

#### Commentaire IV.4 (à finaliser après exécution)

- **Hypothèse** : $\hat\sigma$ aide le réseau quand la volatilité change de régime (2022 surtout).
- **Contrôle** : le gain n'est attribué à la volatilité que si « MLP + σ̂ » bat **à la fois** le MLP original et le témoin permuté, avec un écart supérieur au bruit entre graines.
- **À lire** : (1) le tableau d'ablation et les verdicts automatiques ; (2) le tableau par régime : l'apport est-il concentré dans le tercile de forte volatilité ? (3) la figure du delta en fonction de $\hat\sigma$ : le réseau aplatit-il son delta quand $\hat\sigma$ augmente, comme BS ? (4) le LSTM sans σ̂ se rapproche-t-il du MLP + σ̂ ? Si oui, sa mémoire joue le rôle d'un estimateur de volatilité. Attention : au début de chaque fenêtre, le LSTM n'a encore rien vu, alors que σ̂ résume les 20 jours précédents.
- **Où la volatilité peut ne pas aider** : si la volatilité réalisée **pendant** la fenêtre diffère fortement de σ̂ au départ (choc soudain), ou si le jeu d'entraînement 2015–2020 ne contient pas assez de fenêtres à forte volatilité pour apprendre la relation (en dehors de 2020).
- **Comparaison clé** : le MLP + σ̂ doit être comparé à **BS σ̂ glissante**, qui utilise exactement la même information sans apprentissage.

## Bilan du projet et limites

### Ce que l'on a réellement étudié

1. **Partie I** : reproduction du tableau de l'article sur GBM ; comparaison du réseau avec Black-Scholes, Leland et une règle simple à un paramètre (bande de non-transaction), pour 5 niveaux de frais et 3 graines.
2. **Partie II** : traduction concrète en euros pour un courtier à 1 € par ordre, avec une grille fréquence de couverture × notionnel, et un réseau adapté à chaque notionnel.
3. **Partie III** : comportement hors du régime d'entraînement, avec un backtest chronologique sans fuite et une matrice de transfert présentée comme descriptive.
4. **Partie IV** : effet de la taille des données, du chevauchement des fenêtres, de l'ajout de la volatilité et d'une architecture récurrente (LSTM), avec étude d'ablation et plusieurs graines.

### Limites à présenter à l'oral

- **Sous-jacent** : ETF équipondéré fictif construit à partir des 38 actions présentes dans le fichier, appliquées rétrospectivement à tout l'historique, d'où un **biais du survivant**. La pondération n'est pas celle du vrai CAC 40. Les cours sont a priori ajustés des dividendes.
- **Pas d'options cotées** : le call est fictif ; la prime de marché, le smile de volatilité et les spreads d'options ne sont pas modélisés. $r = 0$.
- **Frais** : proportionnels dans les parties I, III et IV (1 %, niveau élevé repris de l'article) ; forfait de 1 € + 0,1 % (hypothèse) dans la partie II.
- **Dépendance des fenêtres réelles** : les fenêtres glissantes se chevauchent ; quelques dizaines de périodes réellement indépendantes seulement. Les comparaisons sur données réelles sont **descriptives**.
- **Graines** : 3 à 5 graines donnent un ordre de grandeur de la variabilité, pas un intervalle de confiance précis.
- **Hyperparamètres** : architecture et optimiseur fixés comme dans l'article, sans recherche d'hyperparamètres (ce qui peut désavantager le LSTM).

### Conclusion attendue

La question n'est pas d'obtenir systématiquement un meilleur réseau. Il s'agit d'expliquer **dans quels scénarios de frais, de données et de marché** le réseau réduit la dispersion du coût, **avec quel coût moyen et combien d'ordres**, et dans quels cas une règle classique reste préférable. Les conclusions chiffrées doivent être rédigées **après** l'exécution complète du notebook, à partir des tableaux et des lectures automatiques de chaque partie.

### Sources

- Brugière, P. & Turinici, G. (2025), *Model-Free Deep Hedging with Transaction Costs and Light Data Requirements*.
- Leland, H. E. (1985), *Option Pricing and Replication with Transactions Costs*, Journal of Finance.
- Supports de cours et TP du M2 ISF : réseaux denses, convolutionnels, descentes de gradient, réseaux récurrents.
- Tarif Trade Republic : 1 € par ordre (grille tarifaire publique du courtier).